## Version note · v10.1 (hotfix on top of the v10 review fixes)
- **Bug fix (Block 5):** `odc.stac.load(..., groupby="solar_day")` merges scenes acquired on the same day into one time slice, so the v10 per-scene offset array (one entry per scene) could be longer than the merged time axis (e.g. 8 scenes → 7 slices) and crashed with `CoordinateValidationError: conflicting sizes for dimension 'time'`. Per-scene offsets are now keyed by **solar day** (UTC acquisition time shifted by longitude/15 h — the same approximation odc-stac uses for the grouping) and looked up per `ds.time` entry, so a length mismatch is impossible. Same-day scenes with conflicting baselines keep the first offset and warn; any residual lookup falls back to the nearest day (also warned). The scene → solar-day offset map is stored in the composite cache metadata.

### v10 (review fixes on top of v9)
- **Upload validation (Block 3):** files chosen through the upload buttons are now checked by **file name** exactly like session files - a file carrying a different reference year (e.g. a 2012 raster offered to the 2018 dialog) is refused, never re-purposed.
- **Per-scene baseline offset (Block 5):** the +1000 DN Sentinel-2 processing-baseline offset is removed **per scene**, from each scene's own baseline. Mixed baselines can no longer bias the median composite. The composite cache key was bumped, so older caches are rebuilt automatically.
- **Block 8:** the previous NaN check could not fail by construction (Block 6 already removes exactly those pixels). It is now a true consistency check: `lab == 255` must hold exactly where imagery is missing or the label raster has no coverage, with both pixel counts reported.
- **Time budgets kept honest:** Block 10 uses a slim validation (first 128 tiles) when the budget was already hit mid-epoch; Block 11b explicitly falls back to the minimum 2 MC passes (with a warning) when the budget is already spent, instead of silently overrunning.
- **Also:** the exact spatial split is saved to `reports/spatial_split_tiles.npy`; key package versions are recorded in `reports/run_info.json`; the demo-mode warning no longer claims a "reduced area" (the study area is identical in both modes); header updated to v10.

### v9 (review fixes on top of v8 FINAL)
- **Crash fix (Blocks 11 and 11b):** `patches` was used but never imported (only `mpatches` was) → `NameError` at the end of evaluation, after all training. Added `import matplotlib.patches as patches` in Block 1.
- **Block 6:** cloud-mask bias diagnostic now uses the raw labels; before, pixels with no clear observation had already been removed, so the diagnostic understated any bias.
- **Block 8:** replaced the tautological NaN check with a real one (share of labelled / peat pixels lost to composite gaps).
- **Block 11b:** reports how the fine-tuned dropout model compares with the Block-10 U-Net.
- **Block 3:** the CORINE 2012 upload now uses `UPLOAD_WAIT_S` like the 2018 one. **Block 13:** no longer shadows the trained `rf` model.
- Everything else is unchanged from v8.


# 🌿 Peatland ML & CORINE Change Detection Pipeline · v10.1
 Open-data Sentinel-2 + CORINE 2012/2018 · Random Forest vs U-Net · Spatially honest evaluation · Change detection from **real** CORINE only

| | |
|---|---|
| **How to run** | `Runtime → Run all`. Block 3 shows **optional upload buttons** for CORINE 2018 (labels) and CORINE 2012 (change detection). Skip 2018 → labelled WorldCover **proxy**. Skip 2012 → Block 12 is **skipped** (nothing is simulated). For big files, drop them into `/content` (Files panel) or mount Google Drive instead of using the browser button. |
| **Best runtime** | GPU (T4) makes the U-Net ~5× faster; CPU also works (training is time-boxed). |
| **Modes** | `MODE = "demo"` (default) is a fast preset for a live panel; set `MODE = "full"` for the complete run. |
| **Outputs** | `reports/` (metrics with block-bootstrap CIs, 100 m aggregated scores, data-quality report, figures, MC-dropout uncertainty report, and transition matrices when real CORINE 2012 is supplied) + `models/unet_best.pth` + auto-downloaded zip. |

**Pipeline:** 🛰️ Sentinel-2 Download → 🏷️ Labels → 🧩 Tiles → 🗺️ Spatial Split → 🔍 QC → 🌲 Random Forest → 🧠 U-Net → 📊 Evaluation → 🔄 CORINE 2012 vs 2018 Change Detection

**Design principles:** open data only · no synthetic labels anywhere · every number is computed in this run · the pipeline **stops or skips** rather than inventing data.

**How to read the scores:** labels are CORINE (100 m, 25 ha minimum mapping unit), so every F1 / kappa is *agreement with CORINE*, not field-verified accuracy. Confidence intervals come from resampling whole test blocks.

**What changed in v8 (review fixes)**
- Block 12 no longer simulates 2012: it needs a real CORINE 2012 raster (same release as 2018) or it is skipped; stale outputs are deleted.
- Label files are matched by reference year in the file name (2012 can never be used as 2018); upload-widget text bug fixed.
- Evaluation adds block-bootstrap 95% CIs (paired RF-vs-U-Net), 100 m aggregated scores, and context-window U-Net inference.
- Verdict wording is CI-based and flags time-boxed U-Net runs.
- Cloud-mask bias diagnostic for peat (SCL *dark area / shadow*), S2 scene IDs saved, deterministic kernels where available.
- QC: tautological check replaced; class 323 mapped; Blocks 6/7 re-runnable; cosmetic emoji fixes.


## Block 1 · Environment, helpers & reproducibility
**What it does:** installs only missing packages, imports everything, fixes all random seeds, switches PyTorch to deterministic kernels where available, and detects CPU/GPU.

**Why it matters:** a fixed seed and a printed environment line make the run repeatable and auditable. GPU kernels and mixed precision are still not bit-exact, so expect run-to-run differences in the third decimal. The exact Sentinel-2 scene IDs are saved in `run_info.json` so the same inputs can be re-fetched.

> 🎤 **Say to the panel:** *Every run is seeded and records its environment and the exact Sentinel-2 scene IDs, so a colleague can reproduce my numbers within run-to-run noise.*


In [ ]:
import sys, subprocess, importlib, time, os, json, glob, zipfile, shutil, copy, random, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # needed for deterministic cuBLAS

# ---- install only what is missing (fast on Colab) ----
_REQ = {"pystac_client": "pystac-client", "planetary_computer": "planetary-computer", "odc.stac": "odc-stac", "xarray": "xarray",
         "segmentation_models_pytorch": "segmentation-models-pytorch", "rasterio": "rasterio",
         "sklearn": "scikit-learn", "scipy": "scipy", "pyproj": "pyproj", "matplotlib": "matplotlib", "tqdm": "tqdm", "pandas": "pandas", "seaborn": "seaborn"}
def _missing():
    out = []
    for mod, pkg in _REQ.items():
        try: importlib.import_module(mod)
        except Exception: out.append(pkg)
    return out
_m = _missing()
if _m:
    print("Installing:", ", ".join(_m))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _m)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patches as patches   # Blocks 11 / 11b use `patches.` (was an undefined name -> NameError)
from matplotlib.colors import ListedColormap
import rasterio
from rasterio.warp import reproject, Resampling
from pyproj import Transformer
from affine import Affine
import pystac_client, planetary_computer, odc.stac
from odc.geo.geobox import GeoBox
import dask
import xarray as xr
from dask.diagnostics import ProgressBar
from scipy import ndimage as ndi
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, cohen_kappa_score
import torch, torch.nn as nn
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm
import pandas as pd
import seaborn as sns
from IPython.display import display, HTML

# ---- reproducibility ----
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
try: torch.use_deterministic_algorithms(True, warn_only=True)   # warn (not fail) if an op has no deterministic kernel
except Exception: pass
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(os.cpu_count() or 2)
Path("reports").mkdir(exist_ok=True); Path("models").mkdir(exist_ok=True); Path("data/raw").mkdir(parents=True, exist_ok=True)

# ---- presentation helpers ----
def banner(step, title, sub=""):
    display(HTML(f"""<div style="margin:28px 0 12px;padding:14px 20px;border-radius:10px;color:#fff;
      background:linear-gradient(90deg,#0f2027,#203a43,#2c5364);box-shadow:0 3px 8px rgba(0,0,0,.25);font-family:Segoe UI,Arial,sans-serif">
       <span style="background:#4ade80;color:#052e16;border-radius:50%;padding:3px 10px;font-weight:700;margin-right:10px">{step}</span>
       <span style="font-size:20px;font-weight:700">{title}</span>
       <div style="opacity:.8;margin-top:4px;font-size:13px">{sub}</div></div>"""))

def _note(msg, bg, fg, bar, icon):
    display(HTML(f'<div style="padding:9px 14px;margin:6px 0;background:{bg};color:{fg};border-left:5px solid {bar};'
                 f'border-radius:4px;font-family:Segoe UI,Arial,sans-serif;font-size:14px">{icon} {msg}</div>'))
def ok(m):   _note(m, "#dcfce7", "#14532d", "#22c55e", "✅")
def info(m): _note(m, "#e0f2fe", "#0c4a6e", "#0ea5e9", "ℹ️")
def warn(m): _note(m, "#fef3c7", "#78350f", "#f59e0b", "⚠️")

def html_table(rows, header, highlight=None, width="100%"):
    th = " ".join(f'<th style="padding:6px 10px;text-align:left;background:#1e293b;color:#fff">{h}</th>' for h in header)
    body = ""
    for i, r in enumerate(rows):
        bg = "#fef9c3" if highlight is not None and i == highlight else ("#f8fafc" if i % 2 else "#ffffff")
        body += "<tr>" + " ".join(f'<td style="padding:5px 10px;background:{bg};color:#0f172a">{c}</td>' for c in r) + "</tr>"
    display(HTML(f'<table style="border-collapse:collapse;width:{width};font-family:Segoe UI,Arial,sans-serif;font-size:13px;'
                 f'box-shadow:0 1px 4px rgba(0,0,0,.15)"><tr>{th}</tr>{body}</table>'))

TIMINGS = {}
class stage:
    """Context manager that records wall-time per pipeline stage."""
    def __init__(self, name): self.name = name
    def __enter__(self): self.t = time.time(); return self
    def __exit__(self, *a): TIMINGS[self.name] = time.time() - self.t

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU) - Runtime > Change runtime type > T4 GPU makes the U-Net ~5x faster"
display(HTML(f"""<div style="padding:16px;border-radius:12px;background:linear-gradient(135deg,#ecfeff,#dcfce7);font-family:Segoe UI,Arial,sans-serif">
 <h2 style="margin:0;color:#064e3b">🌿 peatland-ml-change-detection · Sentinel-2 + CORINE · RF vs U-Net</h2>
 <div style="margin-top:8px;color:#134e4a;font-size:14px">Pipeline: 🛰️ composite → 🏷️ labels → 🧩 tiles → 🗺️ spatial split → 🔍 QC → 🌲 RF → 🧠 U-Net → 🔄 Change Detection</div>
 <div style="margin-top:6px;color:#475569;font-size:13px">Python {sys.version.split()[0]} · torch {torch.__version__} · CPU cores {os.cpu_count()} · GPU: {gpu} · seed {SEED}</div></div>"""))

## Block 2 · Configuration
**What it does:** one `CFG` dictionary holds dates, bands, tile/block sizes, model settings, bootstrap and aggregation settings, and the CORINE → 7-class lookup tables.

**Why it matters:** no magic numbers hidden in code; changing a setting needs one edit. The class scheme matches the study design (peat bogs = class 4); all 44 CORINE codes are mapped (323 → open semi-natural).

> 🎤 **Say to the panel:** *All choices are in one place; the class mapping is explicit and handles both CORINE raster encodings.*


In [ ]:
# ===================== ALL SETTINGS LIVE HERE (single source of truth) =====================
CFG = dict(
    year=2018, months=("05", "09"), max_cloud=30, max_scenes_per_tile=6,
    bands=["B02", "B03", "B04", "B08", "B11", "B12"], res=10, epsg=32629,
    area_fraction=1.0, min_peat_px=50_000,
    tile=128, block=256, buffer_tiles=1, min_valid_tile=0.30, split=(0.70, 0.15, 0.15),
    rf_max_px=150_000, rf_trees=60,
    unet_batch=16, unet_lr=2e-3, unet_epochs=12, unet_patience=3, unet_min_epochs=5, unet_tiles_per_epoch=640,
    unet_budget_s=600 if torch.cuda.is_available() else 1500,
    unet_context_px=32, boot_n=1000, agg_px=10,
    seed=SEED)

assert (CFG["tile"] + 2 * CFG["unet_context_px"]) % 32 == 0, "tile + 2*context must be divisible by 32 for the U-Net"
MODE = "demo"
if MODE == "demo":
    CFG.update(max_scenes_per_tile=4, rf_max_px=80_000, rf_trees=40,
               unet_budget_s=180 if torch.cuda.is_available() else 420)

AREAS = [dict(name="Irish Midlands bogs",     bbox=(-7.90, 53.10, -7.50, 53.40)),
         dict(name="Connemara blanket bog",   bbox=(-9.90, 53.40, -9.50, 53.60))]

CLASS_NAMES = ["Artificial", "Agriculture", "Forest", "Open semi-natural", "Peat bogs", "Other wetlands", "Water"]
CLASS_COLORS = ["#d62728", "#ffd966", "#2e7d32", "#a8d08d", "#6d4c41", "#26a69a", "#1e88e5", "#ffffff"]
CMAP = ListedColormap(CLASS_COLORS)
PEAT = 4
PS = 3   # plotting stride (display only)

CLC_CODES = [111,112,121,122,123,124,131,132,133,141,142,211,212,213,221,222,223,231,241,242,243,244,311,312,313,
             321,322,323,324,331,332,333,334,335,411,412,421,422,423,511,512,521,522,523]
def _code_to_class(c):
    if 111 <= c <= 142: return 0
    if 211 <= c <= 244: return 1
    if c in (311, 312, 313): return 2
    if c in (321, 322, 323, 324, 331, 332, 333, 334, 335): return 3
    if c == 412: return 4
    if c in (411, 421, 422, 423): return 5
    if 511 <= c <= 523: return 6
    return 255
CODE_LUT = np.full(1024, 255, np.uint8)
for c in CLC_CODES: CODE_LUT[c] = _code_to_class(c)
IDX_LUT = np.full(256, 255, np.uint8)
for i, c in enumerate(CLC_CODES, start=1): IDX_LUT[i] = CODE_LUT[c]

WC_LUT = np.full(256, 255, np.uint8)
for k, v in {50: 0, 40: 1, 10: 2, 20: 3, 30: 3, 60: 3, 100: 3, 90: 4, 95: 5, 80: 6}.items(): WC_LUT[k] = v

banner("2", "Configuration", "one dictionary, no magic numbers hidden in the code")
html_table([["MODE", MODE]] + [[k, str(v)] for k, v in CFG.items() if k not in ("bands",)] + [["bands", ", ".join(CFG["bands"]) + " + NDVI + NDMI"]],
           ["setting", "value"], width="70%")

## Block 3 · Upload widgets · CORINE label files
**What it does:** looks for CORINE 2018 (labels) and CORINE 2012 (change detection) in the session / Google Drive, matching on the **reference year in the file name** (`CLC2018`, `CLC2012`) and never substituting one year for the other. Otherwise it shows optional upload buttons with a countdown and then **continues automatically**. Uploads are validated the same way (v10): a file whose name carries a different reference year is refused, never re-purposed.

**Why it matters:** CORINE needs a free Copernicus login so it cannot be downloaded automatically. Without 2018 a clearly-labelled WorldCover **proxy** is used and flagged in every report; without 2012 the change-detection block is **skipped** - nothing is simulated.

**Tip:** the browser upload holds the whole file in memory and can be slow or crash the tab for the full-Europe raster. Prefer dragging the file into Colab's Files panel (`/content`) or mounting Google Drive, then re-run this cell.

> 🎤 **Say to the panel:** *I never silently substitute data: without CORINE the notebook says so and labels the results as proxy, and without a real 2012 layer it reports no change at all.*


In [ ]:
def _resolve_tif(p, depth=0):
    p = Path(p)
    if p.suffix.lower() in (".tif", ".tiff"): return p
    if p.suffix.lower() == ".zip" and depth < 4:
        out = Path("data/raw") / p.stem; out.mkdir(parents=True, exist_ok=True)
        if not any(out.iterdir()):
            with zipfile.ZipFile(p) as z: z.extractall(out)
        tifs = [t for t in out.rglob("*") if t.suffix.lower() in (".tif", ".tiff") and "ovr" not in t.name.lower()]
        if tifs:
            tifs.sort(key=lambda t: ("clc" not in t.name.lower(), -t.stat().st_size)); return tifs[0]
        for z2 in out.rglob("*.zip"):
            r = _resolve_tif(z2, depth + 1)
            if r: return r
    return None

_SEARCH_DIRS = (Path("."), Path("/content"), Path("data/raw"), Path("/content/drive/MyDrive"))
_OTHER_YEARS = ("1990", "2000", "2006", "2012")

def _clc_candidates():
    seen, out = set(), []
    for d in _SEARCH_DIRS:
        if not d.is_dir(): continue
        for f in sorted(d.iterdir()):
            n = f.name.lower()
            if f.is_file() and f.suffix.lower() in (".tif", ".tiff", ".zip") and any(k in n for k in ("clc", "corine", "u20")):
                r = f.resolve()
                if r not in seen: seen.add(r); out.append(f)
    return out

def _find_clc(year):
    """CORINE file for exactly this reference year. Never falls back to a file of another year."""
    keys = (f"clc{year}", f"clc_{year}", f"corine{year}", f"corine_{year}")
    cands = _clc_candidates()
    for f in cands:
        if any(k in f.name.lower() for k in keys): return f
    if year == 2018:   # a file with no year in its name is accepted for 2018 only if it carries no other reference year
        for f in cands:
            n = f.name.lower().replace("u2018", "")
            if not any(y in n for y in _OTHER_YEARS): return f
    return None

UPLOAD_WAIT_S = 15

def _upload_html(tag, title, wait_s, fallback):
    html = r"""
     <div style="font-family:Segoe UI,Arial,sans-serif;padding:12px 16px;border:2px dashed #0ea5e9;border-radius:10px;background:#f0f9ff;max-width:660px">
       <b>📂 Optional: choose the __TITLE__ file (.tif or .zip)</b>
       <div style="margin:8px 0"><input type="file" id="in-__ID__" accept=".tif,.tiff,.zip"></div>
       <div id="msg-__ID__" style="color:#0c4a6e;font-size:13px">Waiting <span id="t-__ID__">__WAIT__</span>s - then __FALLBACK__.</div>
     </div>
     <script>
    __V__ = {state: "idle", name: "", data: null};
    document.getElementById("in-__ID__").addEventListener("change", function(e) {
      const f = e.target.files[0]; if (!f) return;
      __V__.state = "reading"; __V__.name = f.name;
      document.getElementById("msg-__ID__").textContent = "Reading " + f.name + "...";
      const r = new FileReader();
      r.onload = function() { __V__.data = r.result.split(",")[1]; __V__.state = "done";
        document.getElementById("msg-__ID__").textContent = "Received " + f.name + " - continuing."; };
      r.onerror = function() { __V__.state = "error"; };
      r.readAsDataURL(f);
    });
    let left = __WAIT__;
    const iv = setInterval(function() {
      if (__V__.state !== "idle") { clearInterval(iv); return; }
      left -= 1;
      if (left <= 0) { clearInterval(iv); document.getElementById("msg-__ID__").textContent = "No file chosen - continuing."; }
      else document.getElementById("t-__ID__").textContent = left;
    }, 1000);
     </script>"""
    # named placeholders (NOT a blanket replace of a single letter) so no other text in the HTML/JS can be corrupted
    return (html.replace("__V__", f"window._up_{tag}").replace("__ID__", tag).replace("__TITLE__", title)
                .replace("__FALLBACK__", fallback).replace("__WAIT__", str(int(wait_s))))

_UPLOAD_YEAR_TOKENS = ("1990", "2000", "2006", "2012", "2018")   # every CORINE reference year

def timed_upload(wait_s, tag="clc2018", title="CORINE 2018 (labels)", fallback="the pipeline continues with the labelled WorldCover proxy", year=None):
    if wait_s <= 0: return None
    try:
        import base64
        from google.colab import output
    except ImportError:
        warn("Not running in Colab: place the CORINE file next to the notebook and re-run this cell."); return None
    V = f"window._up_{tag}"
    try:
        display(HTML(_upload_html(tag, title, wait_s, fallback)))
        t0, state = time.time(), "idle"
        while time.time() - t0 < wait_s and state == "idle":
            time.sleep(0.5); state = output.eval_js(f"{V}.state")
        if state == "reading":
            t1 = time.time()
            while state == "reading" and time.time() - t1 < 300:
                time.sleep(0.5); state = output.eval_js(f"{V}.state")
        if state != "done": return None
        name = Path(output.eval_js(f"{V}.name")).name
        if year is not None:   # v10: uploads are validated like session files - the year in the file name must match
            _clash = [y for y in _UPLOAD_YEAR_TOKENS if y != str(year) and y in name.lower()]
            if _clash:
                warn(f"The chosen file <code>{name}</code> carries reference year(s) {_clash} in its name - it will <b>not</b> be used as {year} "
                     f"(the pipeline never substitutes years). Upload a file whose name says {year}, or place it in /content and re-run this cell.")
                return None
        n = int(output.eval_js(f"{V}.data.length")); CH = 2_097_152
        out = Path("data/raw") / (name if tag == "clc2018" else f"corine_2012_{name}")
        with open(out, "wb") as fh:
            for i in range(0, n, CH): fh.write(base64.b64decode(output.eval_js(f"{V}.data.slice({i},{i+CH})")))
        return out
    except Exception as e:
        warn(f"Upload button unavailable ({type(e).__name__}) - continuing automatically."); return None

banner("3", "Upload the CORINE files", "optional upload buttons · matched by reference year · the run continues automatically after the countdown")
label_file, label_path = _find_clc(2018), None
if label_file:
    info(f"Found <b>{label_file}</b> (CORINE 2018 labels) already in the session - skipping the upload dialog (fully automatic).")
else:
    info("Download <b>CORINE Land Cover 2018 (raster, 100 m)</b> from "
         "<a href='https://land.copernicus.eu/en/products/corine-land-cover' target='_blank'>land.copernicus.eu</a> "
         "(free login) - the .zip or .tif is fine. <b>Large file?</b> Drag it into Colab's Files panel (<code>/content</code>) or mount Google Drive "
         "and re-run this cell - the browser button can be slow for the full-Europe raster.")
    label_file = timed_upload(UPLOAD_WAIT_S, "clc2018", "CORINE 2018 (labels)", "the pipeline continues with the labelled WorldCover proxy", year=2018)

if label_file:
    label_path = _resolve_tif(label_file)
    if label_path is None: warn("Could not find a GeoTIFF inside that file - falling back to the proxy labels.")
LABEL_MODE = "corine" if label_path else "worldcover_proxy"
if LABEL_MODE == "corine": ok(f"CORINE 2018 ready: <code>{label_path}</code> ({label_path.stat().st_size/1e6:.1f} MB)")
else: warn("No CORINE file - using <b>ESA WorldCover as a PROXY</b>. It has no peat class (herbaceous wetland is substituted), "
            "so peat-bog results are NOT comparable to CORINE results and are labelled as such in every report.")

# ---- optional CORINE 2012 (only used by Block 12; never simulated) ----
CLC2012_PATH = None
if LABEL_MODE == "corine":
    f2012 = _find_clc(2012)
    if f2012: info(f"Found <b>{f2012}</b> (CORINE 2012) - used by Block 12.")
    else:
        info("<b>Optional - CORINE 2012</b> (same release as the 2018 file; file name containing <code>CLC2012</code>). Only Block 12 (change detection) needs it. "
             "If you skip it, Block 12 is <b>skipped</b> - no change numbers are simulated.")
        f2012 = timed_upload(UPLOAD_WAIT_S, "clc2012", "CORINE 2012 (change detection)", "Block 12 (change detection) is skipped", year=2012)
    if f2012:
        CLC2012_PATH = _resolve_tif(f2012)
        if CLC2012_PATH is None: warn("Could not find a GeoTIFF inside the CORINE 2012 file - Block 12 will be skipped.")
        elif Path(CLC2012_PATH).resolve() == Path(label_path).resolve():
            warn("The 2012 and 2018 files are the same raster - Block 12 will be skipped."); CLC2012_PATH = None
        else: ok(f"CORINE 2012 ready: <code>{CLC2012_PATH}</code>")
else:
    info("Change detection needs real CORINE rasters for both years; with proxy labels Block 12 will be skipped.")


## Block 4 · Study area & labels
**What it does:** builds one 10 m UTM grid (trimmed to whole spatial blocks), warps CORINE onto it, auto-detects index-vs-code encoding and checks peat coverage.

**Why it matters:** imagery and labels share the exact same grid, so misalignment (a classic silent bug) cannot happen.

> 🎤 **Say to the panel:** *I verify the study box really contains peat-bog labels before spending time on imagery.*

In [ ]:
banner("4", "Study area & labels", "grid is built once, labels are warped onto it - imagery and labels can never drift apart")
catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=planetary_computer.sign_inplace)

def make_geobox(bbox, frac=1.0, res=10, epsg=32629, block=256):
    tr = Transformer.from_crs(4326, epsg, always_xy=True)
    pts = [tr.transform(x, y) for x in (bbox[0], bbox[2]) for y in (bbox[1], bbox[3])]
    xs, ys = [p[0] for p in pts], [p[1] for p in pts]
    cx, cy = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2
    hw, hh = (max(xs) - min(xs)) / 2 * frac, (max(ys) - min(ys)) / 2 * frac
    xmin, ymin = np.floor((cx - hw) / res) * res, np.floor((cy - hh) / res) * res
    W = int((2 * hw) / res) // block * block
    H = int((2 * hh) / res) // block * block
    return GeoBox((H, W), Affine(res, 0, xmin, 0, -res, ymin + H * res), f"EPSG:{epsg}")

def gb_lonlat(gb):
    tr = Transformer.from_crs(gb.crs.to_wkt(), 4326, always_xy=True)
    H, W = tuple(gb.shape); a = gb.affine
    pts = [tr.transform(*(a * (c, r))) for c in (0, W) for r in (0, H)]
    return (min(p[0] for p in pts), min(p[1] for p in pts), max(p[0] for p in pts), max(p[1] for p in pts))

def load_corine(path, gb):
    dst = np.zeros(tuple(gb.shape), np.int16)
    with rasterio.open(path) as src:
        reproject(source=rasterio.band(src, 1), destination=dst, dst_transform=gb.affine, dst_crs=gb.crs.to_wkt(),
                  resampling=Resampling.nearest, src_nodata=src.nodata, dst_nodata=0)
    vals = np.unique(dst[dst > 0])
    n_code, n_idx = int(np.isin(vals, CLC_CODES).sum()), int(np.isin(vals, np.arange(1, 45)).sum())
    if n_idx > n_code: scheme, lab = "index(1-44)", IDX_LUT[np.clip(dst, 0, 255)]
    else:              scheme, lab = "code(3-digit)", CODE_LUT[np.clip(dst, 0, 1023)]
    return lab.astype(np.uint8), scheme

def load_worldcover(gb):
    items = list(catalog.search(collections=["esa-worldcover"], bbox=gb_lonlat(gb)).item_collection())
    if not items: raise RuntimeError("No WorldCover items found for this area.")
    def yr(it):
        s = str(it.properties.get("start_datetime") or it.datetime or "0000")
        return int(s[:4]) if s[:4].isdigit() else 0
    best = min({yr(i) for i in items}, key=lambda y: abs(y - CFG["year"]))
    ds = odc.stac.load([i for i in items if yr(i) == best], bands=["map"], geobox=gb, resampling="nearest")
    arr = np.asarray(ds["map"].values); arr = arr[0] if arr.ndim == 3 else arr
    return WC_LUT[arr.astype(np.uint8)], f"worldcover {best}"

rows, chosen, best_try = [], None, None
with stage("labels"):
    for area in AREAS:
        gb_try = make_geobox(area["bbox"], CFG["area_fraction"], CFG["res"], CFG["epsg"], CFG["block"])
        lab_try, scheme = load_corine(label_path, gb_try) if LABEL_MODE == "corine" else load_worldcover(gb_try)
        n_peat = int((lab_try == PEAT).sum()); H, W = tuple(gb_try.shape)
        rows.append([area["name"], f"{W*CFG['res']/1000:.1f} x {H*CFG['res']/1000:.1f} km", scheme, f"{n_peat:,} px ({n_peat*100/1e6:.1f} km2)"])
        cand = dict(area=area, gb=gb_try, lab=lab_try, scheme=scheme, n_peat=n_peat)
        if best_try is None or n_peat > best_try["n_peat"]: best_try = cand
        if n_peat >= CFG["min_peat_px"]: chosen = cand; break
html_table(rows, ["area tried", "size", "label scheme", "peat-bog label pixels"])
if chosen is None:
    if best_try["n_peat"] == 0:
        raise RuntimeError("No peat-bog pixels in any study box - check the label file / bounding boxes. Stopping instead of inventing data.")
    chosen = best_try; warn("Peat coverage is below the target; continuing with the best box - peat metrics will be noisy.")
area, gb, lab_raw, scheme = chosen["area"], chosen["gb"], chosen["lab"], chosen["scheme"]
label_source = "corine" if LABEL_MODE == "corine" else scheme
H, W = tuple(gb.shape)
ok(f"Study area: <b>{area['name']}</b> · grid {H} x {W} px ({H*W/1e6:.1f} M pixels) · CRS {gb.crs} · labels: <b>{label_source}</b> ({scheme})")

## Block 5 · Sentinel-2 median composite
**What it does:** searches Planetary Computer, keeps the clearest scenes per MGRS tile, masks cloud/shadow with the SCL band **per scene**, then takes the median over time.

**Why it matters:** masking before the median removes clouds properly; chunking keeps memory low; limiting scenes cuts download time. It also records, per pixel, how many clear observations there were and how often the scene classification flagged *dark area / cloud shadow* - dark wet peat can be mistaken for shadow, so Block 6 checks whether peat is masked differently from other land. Since v10 the +1000 DN processing-baseline offset is removed **per scene** (each scene corrected from its own baseline), so mixed baselines can no longer bias the composite.

> 🎤 **Say to the panel:** *Cloud masking happens per scene before compositing, not afterwards.*

In [ ]:
banner("5", "Sentinel-2 median composite", "cloud-masked per scene, then median over time · lazy + chunked so it fits in Colab RAM")
def pick_scenes(bbox):
    y = CFG["year"]
    found = catalog.search(collections=["sentinel-2-l2a"], bbox=bbox, datetime=f"{y}-{CFG['months'][0]}-01/{y}-{CFG['months'][1]}-30",
                           query={"eo:cloud_cover": {"lt": CFG["max_cloud"]}}).item_collection()
    by = {}
    for it in found: by.setdefault(it.properties.get("s2:mgrs_tile", "?"), []).append(it)
    sel = []
    for t, its in by.items():
        its.sort(key=lambda i: i.properties.get("eo:cloud_cover", 100)); sel += its[:CFG["max_scenes_per_tile"]]
    return sel, {t: len(v) for t, v in by.items()}, len(found)

import hashlib
def _enc(a):
    return np.where(np.isfinite(a), np.clip(np.rint(a), 0, 65534), 65535).astype(np.uint16)
def _dec(u):
    a = u.astype("float32"); a[u == 65535] = np.nan; return a
_key = hashlib.md5(json.dumps([area["bbox"], list(map(int, gb.shape)), list(gb.affine)[:6],
                               {k: CFG[k] for k in ("year", "months", "max_cloud", "max_scenes_per_tile", "bands", "epsg")}, "v10-per-scene-offset"], default=str).encode()).hexdigest()[:10]
CACHE = Path("data/interim") / f"composite_{_key}.npz"; CACHE.parent.mkdir(parents=True, exist_ok=True)
_cached = CACHE.exists()

with stage("sentinel2"):
    if _cached:
        _z = np.load(CACHE); comp = _dec(_z["comp"]); _m = json.loads(str(_z["meta"]))
        N_SCENES, BASELINES, OFFSET = _m["n_scenes"], _m["baselines"], _m["offset"]
        SCENE_IDS = _m.get("scene_ids", [])
        NOBS, NDARK, NTOT = (_z[k] if k in _z.files else None for k in ("nobs", "ndark", "ntot"))
        info(f"Composite <b>loaded from cache</b> (built earlier in this session from {N_SCENES} scenes, processing baselines {BASELINES}). Delete <code>data/interim</code> to rebuild it from Planetary Computer.")
    else:
        info(f"No cached composite for this configuration (<code>{CACHE.name}</code>). Building it from Planetary Computer now (this is the slow step).")
        sel, per_tile, n_found = pick_scenes(gb_lonlat(gb))
        if not sel: raise RuntimeError("No Sentinel-2 scenes found - widen the dates or cloud limit in CFG.")
        BASELINES = sorted({str(i.properties.get("s2:processing_baseline", "?")) for i in sel})
        def _ge4(b):
            try: return float(b) >= 4.0
            except ValueError: return None
        _flags = {_ge4(b) for b in BASELINES}
        OFFSET = 1000.0 if _flags == {True} else 0.0    # representative value for the log / run_info
        # v10: the +1000 DN offset is corrected PER SCENE from its own processing baseline, so mixed
        # baselines can no longer bias the median composite (v8/v9 used a single global offset)
        _OFFSETS = [1000.0 if _ge4(str(_it.properties.get("s2:processing_baseline", ""))) is True else 0.0 for _it in sel]
        info(f"Sentinel-2 processing baselines in use: {BASELINES} -> per-scene offset correction ({sum(1 for _o in _OFFSETS if _o > 0)}/{len(_OFFSETS)} scenes carry the +1000 DN offset)")
        if len(_flags) > 1: warn("Scenes mix processing baselines. The +1000 DN offset IS corrected per scene (v10); residual radiometric differences between baselines can remain - note this in the limitations.")
        info(f"{n_found} scenes under {CFG['max_cloud']}% cloud across tiles {per_tile} → using the {len(sel)} clearest.")
        ds = odc.stac.load(sel, bands=CFG["bands"] + ["SCL"], geobox=gb, groupby="solar_day", chunks={"x": 1024, "y": 1024},
                           resampling={**{b: "bilinear" for b in CFG["bands"]}, "SCL": "nearest"}, fail_on_error=False)
        refl = ds[CFG["bands"]].to_array("band").astype("float32")
        good = ds["SCL"].isin([4, 5, 6])
        # v10.1 fix: groupby="solar_day" merges scenes acquired on the same day into ONE time
        # slice, so ds.time can be shorter than sel (e.g. 8 scenes -> 7 slices) and the v10
        # positional _OFFSETS no longer line up (CoordinateValidationError on 'time').
        # Instead, key each scene's offset by its SOLAR DAY - UTC acquisition time shifted by
        # longitude/15 h, the same approximation odc-stac uses for the grouping - and look up
        # one offset per ds.time entry, so a length mismatch is impossible.
        def _solar_day(_it):
            _t = _it.datetime or _it.properties.get("start_datetime")
            _t = pd.Timestamp(_t)
            if _t.tzinfo is not None: _t = _t.tz_convert("UTC").tz_localize(None)
            _bb = _it.bbox if _it.bbox is not None else (0.0, 0.0, 0.0, 0.0)
            return (_t + pd.Timedelta(hours=((_bb[0] + _bb[2]) / 2.0) / 15.0)).normalize()
        _DAY2OFF = {}
        for _it, _off in zip(sel, _OFFSETS):
            _d = _solar_day(_it)
            if _d in _DAY2OFF and _DAY2OFF[_d] != _off:
                warn(f"Two scenes share solar day {_d.date()} but differ in baseline offset ({_DAY2OFF[_d]:.0f} vs {_off:.0f} DN); using {_DAY2OFF[_d]:.0f} DN for the merged slice.")
                continue
            _DAY2OFF.setdefault(_d, _off)
        if len(refl.coords["time"]) != len(_OFFSETS):
            info(f"groupby='solar_day' merged {len(_OFFSETS)} scenes into {len(refl.coords['time'])} time slices; offset per solar day: "
                 + ", ".join(f"{k.date()}:{v:.0f}" for k, v in sorted(_DAY2OFF.items())))
        _off_vals, _fallbacks = [], []
        for _t in refl.coords["time"].values:
            _d = pd.Timestamp(_t).normalize()
            if _d not in _DAY2OFF:
                _d = min(_DAY2OFF, key=lambda _x: abs((_x - _d).days))
                _fallbacks.append(f"{pd.Timestamp(_t).date()}→{_d.date()} ({_DAY2OFF[_d]:.0f} DN)")
            _off_vals.append(_DAY2OFF[_d])
        if _fallbacks:
            warn("Some time slices had no exact solar-day baseline match; nearest-day offset used: " + "; ".join(_fallbacks))
        _off_da = xr.DataArray(np.asarray(_off_vals, "float32"), dims=("time",), coords={"time": refl.coords["time"]})
        masked = refl.where(good & (refl > 0)) - _off_da     # per-solar-day offset, broadcast along time
        # diagnostics computed in the SAME dask pass: clear observations per pixel, and how often SCL flagged
        # "dark area" / "cloud shadow" (dark wet peat can be mistaken for shadow -> possible composite bias)
        med = masked.chunk({"time": -1, "band": 1}).median("time", skipna=True)
        n_clear = good.sum("time").astype("uint8")
        n_dark = ds["SCL"].isin([2, 3]).sum("time").astype("uint8")
        n_tot = (ds["SCL"] > 0).sum("time").astype("uint8")
        with ProgressBar():
            comp, NOBS, NDARK, NTOT = [a.values for a in dask.compute(med, n_clear, n_dark, n_tot)]
        SCENE_IDS = [i.id for i in sel]
        del ds, refl, good, masked, med, n_clear, n_dark, n_tot
        N_SCENES = len(sel)
        _u = _enc(comp); comp = _dec(_u)
        np.savez(CACHE, comp=_u, nobs=NOBS, ndark=NDARK, ntot=NTOT,
                 meta=json.dumps(dict(n_scenes=N_SCENES, baselines=BASELINES, offset=OFFSET, offsets=_OFFSETS, offsets_by_solar_day={str(k.date()): v for k, v in _DAY2OFF.items()}, scene_ids=SCENE_IDS)))
cover = float(np.isfinite(comp).all(0).mean())
ok(f"Composite {'loaded from cache' if _cached else 'built'} in {TIMINGS['sentinel2']:.0f} s · {N_SCENES} scenes · valid coverage {cover*100:.1f}%")

## Block 6 · Feature engineering & label cleaning
**What it does:** scales reflectance, adds NDVI and NDMI (8 channels), marks pixels without imagery as nodata (255) and plots imagery next to labels.

**Why it matters:** NDMI (moisture) is informative for wet peat; nodata is ignored by every model and metric. A cloud-mask bias diagnostic compares peat with other land.

> 🎤 **Say to the panel:** *Eight channels: six bands plus two indices; nodata is excluded everywhere.*

In [ ]:
banner("6", "Feature engineering & label cleaning", "reflectance scaling, NDVI/NDMI, nodata = 255")
with stage("features"):
    refl = np.clip(comp / 10000.0, 0, 1.5).astype("float32")
    B = {n: refl[i] for i, n in enumerate(CFG["bands"])}
    ndvi = (B["B08"] - B["B04"]) / (B["B08"] + B["B04"] + 1e-6)
    ndmi = (B["B08"] - B["B11"]) / (B["B08"] + B["B11"] + 1e-6)
    X = np.concatenate([refl, ndvi[None], ndmi[None]]).astype("float32")
    FEATURES = CFG["bands"] + ["NDVI", "NDMI"]
    finite = np.isfinite(X).all(0)
    lab = lab_raw.copy(); lab[~finite] = 255

# ---- cloud-mask bias diagnostic: is peat masked differently from other land? ----
MASK_DIAG = None
if NOBS is not None:
    # use lab_raw (not `lab`): `lab` already has composite-gap pixels set to 255, which would drop exactly the
    # pixels most affected by masking and make the diagnostic look better than it is
    pm = (lab_raw == PEAT); om = (lab_raw != 255) & ~pm
    dark_share = NDARK.astype("float32") / np.maximum(NTOT, 1)
    rows_d = [[nm, f"{mk.sum():,}", f"{np.median(NOBS[mk]):.0f}", f"{dark_share[mk].mean()*100:.1f}%"]
              for nm, mk in (("Peat bogs", pm), ("All other classes", om)) if mk.any()]
    html_table(rows_d, ["pixels of", "labelled px", "median clear observations", "share of scenes flagged dark/shadow (SCL 2,3)"], width="80%")
    if pm.any() and om.any():
        d_p, d_o = float(dark_share[pm].mean()), float(dark_share[om].mean())
        n_p, n_o = float(np.median(NOBS[pm])), float(np.median(NOBS[om]))
        MASK_DIAG = dict(peat_dark_share=d_p, other_dark_share=d_o, peat_median_clear_obs=n_p, other_median_clear_obs=n_o)
        if (d_p > 1.5 * d_o and d_p > 0.05) or n_p < 0.7 * n_o:
            warn("<b>Cloud-mask bias possible:</b> peat pixels are flagged dark/shadow more often, or have fewer clear observations, than other land. "
                 "The median composite over peat may be built from fewer or biased scenes - list this as a limitation.")
        else: ok("Cloud masking treats peat and other land similarly (dark/shadow share and clear-observation count are comparable).")
else:
    info("Cloud-mask diagnostic unavailable (composite cache built by an older version). Delete <code>data/interim</code> and re-run Block 5 to enable it.")

def stretch(a):
    lo, hi = np.nanpercentile(a, (2, 98)); return np.clip((a - lo) / (hi - lo + 1e-9), 0, 1)
rgb = np.stack([stretch(B["B04"]), stretch(B["B03"]), stretch(B["B02"])], -1)
s = 3
fig, ax = plt.subplots(1, 2, figsize=(15, 6.2))
ax[0].imshow(np.nan_to_num(rgb[::s, ::s]), interpolation="nearest"); ax[0].set_title("Sentinel-2 composite (true colour)", fontweight="bold")
ax[1].imshow(np.where(lab == 255, 7, lab)[::s, ::s], cmap=CMAP, vmin=0, vmax=7, interpolation="nearest")
ax[1].set_title(f"7-class labels ({label_source})", fontweight="bold")
for a in ax: a.axis("off")
ax[1].legend(handles=[mpatches.Patch(color=CLASS_COLORS[i], label=CLASS_NAMES[i]) for i in range(7)], loc="lower left", fontsize=8, framealpha=.9)
plt.tight_layout(); plt.savefig("reports/01_study_area.png", dpi=130); plt.show()

cnt = np.bincount(lab[lab != 255], minlength=7)
html_table([[f"{i}", CLASS_NAMES[i], f"{cnt[i]:,}", f"{cnt[i]/max(cnt.sum(),1)*100:.2f}%"] for i in range(7)],
           ["id", "class", "pixels", "share"], highlight=PEAT, width="60%")

## Block 7 · Tiling & spatial block split
**What it does:** cuts 128×128 tiles nested exactly inside 2.56 km blocks, assigns **whole blocks** to train/val/test (70/15/15), adds a one-tile exclusion ring around test and validation, and standardises features with training statistics only.

**Why it matters:** random pixel splits leak information through spatial autocorrelation and inflate scores; block splits with a buffer give a realistic estimate of performance on new areas.

> 🎤 **Say to the panel:** *I split by spatial block with a buffer to avoid leakage, and normalise with training statistics only.*

In [ ]:
banner("7", "Tiling & spatial block split", "whole 2.56 km blocks go to train / val / test · exclusion ring between them · leakage reduced by whole-block assignment and an exclusion ring")
T, BK = CFG["tile"], CFG["block"]; TPB = BK // T
nTr, nTc = H // T, W // T
with stage("split"):
    valid_frac = (lab != 255).reshape(nTr, T, nTc, T).mean((1, 3))
    tile_ok = valid_frac >= CFG["min_valid_tile"]
    block_ok = tile_ok.reshape(H // BK, TPB, W // BK, TPB).any((1, 3))
    peat_tile = (lab == PEAT).reshape(nTr, T, nTc, T).sum((1, 3))
    ub = np.argwhere(block_ok)

    def assign(seed):
        rng = np.random.default_rng(seed); perm = rng.permutation(len(ub))
        n_te = max(1, round(CFG["split"][2] * len(ub))); n_va = max(1, round(CFG["split"][1] * len(ub)))
        bs = np.full(block_ok.shape, -1, np.int8)
        for k, i in enumerate(perm): bs[tuple(ub[i])] = 2 if k < n_te else (1 if k < n_te + n_va else 0)
        ts = np.kron(bs, np.ones((TPB, TPB), np.int8)); ts[~tile_ok] = -1
        st = np.ones((3, 3), bool)
        near_test = ndi.binary_dilation(ts == 2, st, iterations=CFG["buffer_tiles"])
        ts[near_test & ((ts == 0) | (ts == 1))] = 3
        near_val = ndi.binary_dilation(ts == 1, st, iterations=CFG["buffer_tiles"])
        ts[near_val & (ts == 0)] = 3
        return bs, ts

    def peat_balance(ts):
        kept = peat_tile[(ts >= 0) & (ts < 3)].sum()
        if kept == 0: return False, (np.nan,) * 3
        sh = tuple(peat_tile[ts == k].sum() / kept for k in (0, 1, 2))
        return (sh[0] >= .50 and sh[1] >= .07 and sh[2] >= .07), sh

    chosen_seed, bsplit, TS, shares = None, None, None, None
    for trial in range(CFG["seed"], CFG["seed"] + 300):
        bs, ts = assign(trial); good_split, sh = peat_balance(ts)
        if good_split: chosen_seed, bsplit, TS, shares = trial, bs, ts, sh; break
    if TS is None: chosen_seed = CFG["seed"]; bsplit, TS = assign(chosen_seed); shares = peat_balance(TS)[1]; warn("No perfectly peat-balanced split found; using the default seed.")

    SPLITPIX = np.kron(TS, np.ones((T, T), np.int8))
    np.save("reports/spatial_split_tiles.npy", TS)   # v10: exact split preserved with the reports (pixels: kron(TS, ones(T,T)))
    tr_pix = (SPLITPIX == 0) & (lab != 255)
    mu = X[:, tr_pix].mean(1); sd = X[:, tr_pix].std(1) + 1e-6
    Xn = ((X - mu[:, None, None]) / sd[:, None, None]).astype("float32")
    Xn[~np.isfinite(Xn)] = 0   # nodata -> feature mean (0 after standardisation); ignored by every loss/metric, but keeps U-Net context windows well-defined

names = {0: "Train", 1: "Validation", 2: "Test", 3: "Buffer (excluded)"}
rows = [[names[k], f"{(TS==k).sum():,}", f"{(TS==k).sum()*T*T/1e6:.2f} M px", (f"{shares[k]*100:.1f}%" if k < 3 and not np.isnan(shares[k]) else "-")] for k in range(4)]
html_table(rows, ["split", "tiles (128 x 128)", "area", "share of peat pixels kept"])
ok(f"Block-split seed {chosen_seed} (chosen so train, val and test all contain peat - decided from labels only, never from model scores)")

fig, ax = plt.subplots(figsize=(8.5, 7))
cm = ListedColormap(["#ffffff", "#1f77b4", "#2ca02c", "#d62728", "#9ca3af"])
ax.imshow(TS + 1, cmap=cm, vmin=0, vmax=4, interpolation="nearest"); ax.set_title(f"Spatial split (each cell = 1.28 km tile) · seed {chosen_seed}", fontweight="bold")
ax.legend(handles=[mpatches.Patch(color=c, label=l) for c, l in zip(["#1f77b4", "#2ca02c", "#d62728", "#9ca3af"], ["Train", "Validation", "Test", "Buffer"])], loc="lower right")
ax.axis("off"); plt.tight_layout(); plt.savefig("reports/02_spatial_split.png", dpi=130); plt.show()
# free big temporaries but KEEP `comp`, so Block 6 can be re-run; pop() makes this cell safely re-runnable
for _n in ("refl", "B", "ndvi", "ndmi", "rgb"): globals().pop(_n, None)

## Block 8 · Data-quality gate
**What it does:** recomputes independent checks from the final arrays (CRS, resolution, shapes, NaNs, block purity, buffer distance, class presence, nodata) and writes `reports/data_quality.json`.

**Why it matters:** critical failures stop the run; softer issues (e.g. a class absent from a split) are surfaced as warnings instead of being hidden. v8 replaced a nodata check that could never fail with meaningful ones (composite coverage, train-validation buffer, number of test blocks and test peat pixels). v10 turned the remaining NaN check into a true imagery-vs-label consistency check (with pixel counts), since it could not fail by construction.

> 🎤 **Say to the panel:** *Quality control is automated and fails loudly; the report is committed with the results.*

In [ ]:
banner("8", "Data-quality gate", "independent checks recomputed from the final arrays · pipeline stops if a critical check fails")
checks = []
def chk(name, passed, detail, critical=True):
    checks.append(dict(name=name, status="pass" if passed else ("fail" if critical else "warn"), detail=detail))

res_x, res_y = abs(gb.affine.a), abs(gb.affine.e)
chk("CRS is projected (UTM 29N) and shared by imagery & labels", str(gb.crs).upper().endswith(str(CFG["epsg"])), str(gb.crs))
chk("Pixel size is 10 m", abs(res_x - CFG["res"]) < 1e-6 and abs(res_y - CFG["res"]) < 1e-6, f"{res_x} x {res_y} m")
chk("Image and label shapes match", X.shape[1:] == lab.shape == tuple(gb.shape), f"{X.shape} vs {lab.shape}")
# v10: real consistency check instead of a tautology - lab==255 must hold EXACTLY where imagery is missing or labels absent
_exp_nodata = ~np.isfinite(X).all(0) | (lab_raw == 255)
chk("nodata mask consistent (lab==255 exactly where imagery is missing or labels absent)", bool(np.array_equal(lab == 255, _exp_nodata)),
    f"{int((np.isfinite(X).all(0) & (lab_raw == 255)).sum()):,} px have imagery but no label coverage · "
    f"{int(((lab_raw != 255) & ~np.isfinite(X).all(0)).sum()):,} px have labels but no composite")
_lost = float(((lab_raw != 255) & (lab == 255)).sum() / max((lab_raw != 255).sum(), 1))
_lost_peat = float(((lab_raw == PEAT) & (lab == 255)).sum() / max((lab_raw == PEAT).sum(), 1))
chk("Cloud/composite gaps remove <= 10% of labelled pixels (and of peat pixels)", _lost <= 0.10 and _lost_peat <= 0.10,
    f"{_lost*100:.1f}% of labelled px, {_lost_peat*100:.1f}% of peat px have no valid composite", critical=False)
chk("Labels only contain classes 0-6 or 255", bool(np.isin(np.unique(lab), list(range(7)) + [255]).all()), str(np.unique(lab).tolist()))

v = TS.reshape(H // BK, TPB, W // BK, TPB).transpose(0, 2, 1, 3).reshape(H // BK, W // BK, -1)
m = (v >= 0) & (v <= 2)
mn, mx = np.where(m, v, 9).min(-1), np.where(m, v, -9).max(-1)
chk("No spatial block appears in two splits", bool(((mn == mx) | (~m.any(-1))).all()), f"{int(m.any(-1).sum())} blocks checked")
dist = ndi.distance_transform_cdt(TS != 2, metric="chessboard")
min_gap = int(dist[TS == 0].min()) if (TS == 0).any() else -1
chk("Train tiles are >= buffer distance from every test tile", min_gap >= CFG["buffer_tiles"] + 1, f"minimum gap = {min_gap} tiles (required {CFG['buffer_tiles']+1})")
chk("Train, validation and test are all non-empty", all((TS == k).any() for k in (0, 1, 2)), f"{[(int((TS==k).sum())) for k in (0,1,2)]} tiles")

split_counts = {}
for k, nm in ((0, "train"), (1, "val"), (2, "test")):
    sel = (SPLITPIX == k) & (lab != 255)
    split_counts[nm] = np.bincount(lab[sel], minlength=7).tolist()
for nm in ("train", "val", "test"):
    chk(f"Peat bogs present in {nm}", split_counts[nm][PEAT] > 0, f"{split_counts[nm][PEAT]:,} px", critical=False)
    missing = [CLASS_NAMES[i] for i in range(7) if split_counts[nm][i] == 0]
    chk(f"All 7 classes present in {nm}", not missing, "missing: " + (", ".join(missing) if missing else "none"), critical=False)
chk("Composite has imagery in all 6 bands over >= 90% of the study grid", cover >= 0.90, f"{cover*100:.1f}% of pixels", critical=False)
min_gap_v = int(ndi.distance_transform_cdt(TS != 1, metric="chessboard")[TS == 0].min()) if ((TS == 1).any() and (TS == 0).any()) else -1
chk("Train tiles are >= buffer distance from every validation tile", min_gap_v >= CFG["buffer_tiles"] + 1, f"minimum gap = {min_gap_v} tiles (required {CFG['buffer_tiles']+1})")
n_test_blocks_qc = int((bsplit == 2).sum())
chk("At least 10 test blocks (needed for meaningful block-bootstrap CIs)", n_test_blocks_qc >= 10, f"{n_test_blocks_qc} test blocks", critical=False)
chk("At least 5,000 peat pixels in the test split", split_counts["test"][PEAT] >= 5000, f"{split_counts['test'][PEAT]:,} px", critical=False)

icon = {"pass": "✅", "warn": "⚠️", "fail": "❌"}
html_table([[icon[c["status"]], c["name"], c["detail"]] for c in checks], [" ", "check", "result"])
status = "fail" if any(c["status"] == "fail" for c in checks) else ("warn" if any(c["status"] == "warn" for c in checks) else "pass")
json.dump(dict(status=status, label_source=label_source, label_scheme=scheme, area=area["name"], seed=chosen_seed,
               grid=[H, W], cloud_mask_diagnostic=MASK_DIAG, checks=checks, class_counts_per_split={k: dict(zip(CLASS_NAMES, v)) for k, v in split_counts.items()}),
          open("reports/data_quality.json", "w"), indent=2)
if status == "fail": raise AssertionError("Data-quality gate FAILED - see table above and reports/data_quality.json")
ok(f"Quality gate: <b>{status.upper()}</b> · report saved to reports/data_quality.json")

## Block 9 · Random Forest baseline
**What it does:** trains on square-root-balanced pixel samples - the same effective class prior as the U-Net's square-root class weights, so the two models are compared fairly and predicts all unseen test pixels in batches.

**Why it matters:** a strong, interpretable baseline - the U-Net has to beat it to justify its cost.

> 🎤 **Say to the panel:** *The baseline uses only spectral information, so any U-Net gain reflects spatial context.*

In [ ]:
banner("9", "Random Forest baseline", "per-pixel spectral model · square-root-balanced sampling (same effective class prior as the U-Net)")
TEST_MASK = (SPLITPIX == 2) & (lab != 255)
with stage("random_forest"):
    rng = np.random.default_rng(CFG["seed"])
    flat_lab, flat_tr = lab.ravel(), tr_pix.ravel()
    picks = []
    n_max = int(np.bincount(flat_lab[flat_tr], minlength=7).max())
    for c in range(7):
        idx = np.flatnonzero(flat_tr & (flat_lab == c))
        if len(idx):
            k = max(1, min(len(idx), int(CFG["rf_max_px"] * np.sqrt(len(idx) / n_max))))
            picks.append(rng.choice(idx, k, replace=False))
    picks = np.concatenate(picks)
    Xf = Xn.reshape(len(FEATURES), -1)
    rf = RandomForestClassifier(n_estimators=CFG["rf_trees"], max_depth=18, min_samples_leaf=3, class_weight=None,
                                n_jobs=-1, random_state=CFG["seed"]).fit(Xf[:, picks].T, flat_lab[picks])
    te_idx = np.flatnonzero(TEST_MASK.ravel())
    pred_rf = np.empty(len(te_idx), np.uint8)
    for i in range(0, len(te_idx), 400_000):
        pred_rf[i:i + 400_000] = rf.predict(Xf[:, te_idx[i:i + 400_000]].T)
y_true = flat_lab[te_idx]
ok(f"Random Forest trained on {len(picks):,} sampled pixels and applied to {len(te_idx):,} unseen test pixels in {TIMINGS['random_forest']:.0f} s")
imp = sorted(zip(FEATURES, rf.feature_importances_), key=lambda t: -t[1])
info("Feature importance: " + " · ".join(f"<b>{n}</b> {v*100:.0f}%" for n, v in imp[:5]))

## Block 10 · U-Net (ResNet-18 encoder)
**What it does:** trains a slim U-Net from scratch with weighted cross-entropy (nodata ignored), flip/rotation augmentation, early stopping on validation loss and a hard time budget. At test time each 128 px tile is predicted from a window with `unet_context_px` (32) pixels of surrounding imagery (reflect-padded at the map edge) and cropped back to the tile, so tile edges are not penalised. Only imagery is used as context, never labels.

**Why it matters:** class weights counter the rarity of peat; the time budget keeps the demo predictable; the best checkpoint by validation loss is restored. If the budget is hit, the RF-vs-U-Net comparison is *time-boxed* and the summary says so.

> 🎤 **Say to the panel:** *Early stopping and checkpointing are driven by validation blocks, never by the test blocks.*


In [ ]:
banner("10", "U-Net (ResNet-18 encoder)", "trained from scratch · weighted cross-entropy ignoring nodata · flips + 90° rotations · time-boxed with early stopping")
def gather(code):
    idx = np.argwhere(TS == code)
    if len(idx) == 0: return None, None, idx
    xs = np.stack([Xn[:, r*T:(r+1)*T, c*T:(c+1)*T] for r, c in idx])
    ys = np.stack([lab[r*T:(r+1)*T, c*T:(c+1)*T] for r, c in idx]).astype(np.int64)
    return torch.from_numpy(xs), torch.from_numpy(ys), idx

with stage("unet"):
    Xtr, Ytr, _ = gather(0); Xva, Yva, _ = gather(1)
    cnt = np.bincount(Ytr[Ytr != 255].numpy().ravel(), minlength=7).astype(np.float64)
    w = np.where(cnt > 0, (cnt.sum() / (7 * np.maximum(cnt, 1))) ** 0.5, 0.0); w = w / w[cnt > 0].mean()
    crit = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=DEVICE), ignore_index=255)

    model = smp.Unet("resnet18", encoder_weights=None, in_channels=len(FEATURES), classes=7,
                     decoder_channels=(128, 64, 32, 16, 16)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["unet_lr"], weight_decay=1e-4)
    use_amp = DEVICE.type == "cuda"
    try: scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except Exception: scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    g = torch.Generator().manual_seed(CFG["seed"]); arng = np.random.default_rng(CFG["seed"])
    BS, hist, best, bad, best_state, t0 = CFG["unet_batch"], [], float("inf"), 0, None, time.time()

    def augment(xb, yb):
        k = int(arng.integers(4)); xb, yb = torch.rot90(xb, k, (2, 3)), torch.rot90(yb, k, (1, 2))
        if arng.random() < .5: xb, yb = torch.flip(xb, (3,)), torch.flip(yb, (2,))
        return xb, yb

    rows_live = []; budget_hit = False
    for ep in range(1, CFG["unet_epochs"] + 1):
        model.train(); perm = torch.randperm(len(Xtr), generator=g)[:CFG["unet_tiles_per_epoch"]]; tl, nb = 0.0, 0
        for i in range(0, len(perm), BS):
            ib = perm[i:i + BS]; xb, yb = augment(Xtr[ib], Ytr[ib]); xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp): loss = crit(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tl += float(loss); nb += 1
            if time.time() - t0 > CFG["unet_budget_s"]: budget_hit = True; break
        model.eval(); vl, vn = 0.0, 0
        _va_n = len(Xva) if not budget_hit else min(len(Xva), 128)   # v10: slim validation once the budget is already spent
        with torch.no_grad():
            for i in range(0, _va_n, 32):
                xb, yb = Xva[i:i + 32].to(DEVICE), Yva[i:i + 32].to(DEVICE)
                with torch.autocast(device_type=DEVICE.type, enabled=use_amp): vl += float(crit(model(xb), yb)) * len(xb)
                vn += len(xb)
        tl, vl = tl / max(nb, 1), vl / max(vn, 1); hist.append((tl, vl))
        flag = ""
        if vl < best - 1e-4: best, bad, best_state, flag = vl, 0, copy.deepcopy(model.state_dict()), " ⭐ best"
        else: bad += 1
        print(f"epoch {ep:2d} | train {tl:.3f} | val {vl:.3f} | {time.time()-t0:5.0f}s{flag}")
        if bad >= CFG["unet_patience"] and ep >= CFG["unet_min_epochs"]: print("early stopping (validation loss stopped improving)"); break
        if time.time() - t0 > CFG["unet_budget_s"]: budget_hit = True; print("time budget reached - this is NOT a fixed-epoch run" + (" (last validation used a 128-tile subset)" if _va_n != len(Xva) else "")); break
    if best_state is None: best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state); torch.save(best_state, "models/unet_best.pth")

    P = CFG["unet_context_px"]                      # pixels of surrounding IMAGERY (never labels) around each test tile
    te_tiles = np.argwhere(TS == 2); pred_map = np.full((H, W), 255, np.uint8); model.eval()
    def _window(r, c):
        r0, r1, c0, c1 = r*T - P, (r+1)*T + P, c*T - P, (c+1)*T + P
        patch = Xn[:, max(r0, 0):min(r1, H), max(c0, 0):min(c1, W)]
        pad = ((0, 0), (max(-r0, 0), max(r1 - H, 0)), (max(-c0, 0), max(c1 - W, 0)))
        return np.pad(patch, pad, mode="reflect") if any(p for pp in pad for p in pp) else patch
    with torch.no_grad():
        for i in range(0, len(te_tiles), 16):
            chunk = te_tiles[i:i + 16]
            xb = torch.from_numpy(np.stack([_window(r, c) for r, c in chunk])).to(DEVICE)
            out = model(xb).argmax(1).cpu().numpy().astype(np.uint8)[:, P:P + T, P:P + T]      # crop back to the tile
            for (r, c), p in zip(chunk, out): pred_map[r*T:(r+1)*T, c*T:(c+1)*T] = p
    pred_unet = pred_map.ravel()[te_idx]
ok(f"U-Net trained for {len(hist)} epochs in {TIMINGS['unet']:.0f} s · best validation loss {best:.3f} · checkpoint saved to models/unet_best.pth")
if len(hist) < CFG["unet_min_epochs"]: warn(f"U-Net ran only {len(hist)} epochs (minimum {CFG['unet_min_epochs']}) - it is under-trained; BatchNorm statistics have not settled. Use a GPU runtime or raise unet_budget_s.")
if budget_hit: warn("The U-Net stopped because it hit its <b>time budget</b>: the RF-vs-U-Net comparison is time-boxed, not an architecture verdict. Raise <code>unet_budget_s</code> / use a GPU for a fair comparison.")
fig, ax = plt.subplots(figsize=(6, 3.4)); ax.plot([h[0] for h in hist], "o-", label="train"); ax.plot([h[1] for h in hist], "s-", label="validation")
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("weighted CE loss (log scale)"); ax.set_title("U-Net training curve", fontweight="bold"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("reports/03_unet_training.png", dpi=130); plt.show()

## Block 11 · Evaluation on unseen blocks
**What it does:** scores both models on the **same** test pixels: accuracy, macro F1, kappa, per-class precision/recall/F1, row-normalised confusion matrices, test maps and a peat hit/miss/false-alarm map. New in v8: (1) **95% confidence intervals** from resampling whole test blocks (paired for the RF-vs-U-Net difference); (2) a second scoring on a **100 m grid** (modal class per 10×10-pixel cell) that matches CORINE's native resolution.

**Why it matters:** the labels are 100 m blocks nearest-resampled to 10 m, so pixel scores partly measure how well a model reproduces CORINE's blockiness. The 100 m scores and the intervals show how much of a gap is real. Intervals cover test-block sampling only - not training randomness or other possible splits - and neighbouring blocks are spatially correlated, so treat them as optimistic.

> 🎤 **Say to the panel:** *Scores are agreement with CORINE, with block-bootstrap intervals; if the interval of the RF-vs-U-Net difference spans zero, I do not claim a winner.*


In [ ]:
banner("11", "Evaluation on unseen spatial blocks", "identical test pixels · block-bootstrap 95% CIs · 100 m aggregated scoring · peat reported separately")
LABS = list(range(7))

def metrics_from_cm(cm):
    """All headline metrics from a 7x7 confusion matrix (rows = reference, columns = prediction)."""
    cm = np.asarray(cm, dtype=np.float64)
    tp, sup, prd, n = np.diag(cm), cm.sum(1), cm.sum(0), cm.sum()
    pr = np.divide(tp, prd, out=np.zeros_like(tp), where=prd > 0)
    rc = np.divide(tp, sup, out=np.zeros_like(tp), where=sup > 0)
    f1 = np.divide(2 * pr * rc, pr + rc, out=np.zeros_like(tp), where=(pr + rc) > 0)
    present = sup > 0
    acc = tp.sum() / n if n else 0.0
    pe = float((sup * prd).sum() / n ** 2) if n else 0.0
    kappa = (acc - pe) / (1 - pe) if pe < 1 else 0.0
    has_peat = sup[PEAT] > 0
    return dict(accuracy=float(acc), macro_f1=float(f1[present].mean()) if present.any() else 0.0, kappa=float(kappa),
                peat_f1=float(f1[PEAT]) if has_peat else None, peat_precision=float(pr[PEAT]) if has_peat else None,
                peat_recall=float(rc[PEAT]) if has_peat else None, peat_support_px=int(sup[PEAT]),
                per_class={CLASS_NAMES[i]: dict(precision=float(pr[i]), recall=float(rc[i]), f1=float(f1[i]), support=int(sup[i])) for i in range(7)})

def score(y, p): return metrics_from_cm(confusion_matrix(y, p, labels=LABS))

def block_cms(y, p, blk):
    """Per-spatial-block confusion matrices -> array (n_blocks, 7, 7)."""
    ub, inv = np.unique(blk, return_inverse=True)
    flat = inv.astype(np.int64) * 49 + y.astype(np.int64) * 7 + p.astype(np.int64)
    return np.bincount(flat, minlength=len(ub) * 49).reshape(len(ub), 7, 7)

def block_bootstrap(cms_by_model, n_boot, seed):
    """Resample whole test blocks with replacement. The SAME draws are used for every model, so differences are paired."""
    rng = np.random.default_rng(seed); nb_ = next(iter(cms_by_model.values())).shape[0]
    keys = ["accuracy", "macro_f1", "kappa", "peat_f1", "peat_precision", "peat_recall"]
    draws = {m: {k: [] for k in keys} for m in cms_by_model}
    for _ in range(n_boot):
        w = np.bincount(rng.integers(0, nb_, nb_), minlength=nb_)      # how often each block is drawn
        for m, cms in cms_by_model.items():
            r = metrics_from_cm(np.tensordot(w, cms, axes=1))
            for k in keys: draws[m][k].append(np.nan if r[k] is None else r[k])
    return {m: {k: np.array(v) for k, v in d.items()} for m, d in draws.items()}

def ci(a, lo=2.5, hi=97.5):
    a = a[np.isfinite(a)]
    return (float("nan"),) * 3 if len(a) < 20 else (float(np.mean(a)), float(np.percentile(a, lo)), float(np.percentile(a, hi)))

def aggregate_modal(map_, mask, k):
    """Modal class on a k x k pixel grid; also returns which cells lie completely inside `mask`."""
    Hc, Wc = (map_.shape[0] // k) * k, (map_.shape[1] // k) * k
    full = mask[:Hc, :Wc].reshape(Hc // k, k, Wc // k, k).all((1, 3))
    counts = np.stack([(map_[:Hc, :Wc] == c).reshape(Hc // k, k, Wc // k, k).sum((1, 3)) for c in range(7)])
    return counts.argmax(0).astype(np.uint8), full

# ---- pixel-level scores on the identical test pixels ----
pred_rf_map = np.full(H * W, 255, np.uint8); pred_rf_map[te_idx] = pred_rf; pred_rf_map = pred_rf_map.reshape(H, W)
res = {"RandomForest": score(y_true, pred_rf), "UNet": score(y_true, pred_unet)}
if res["UNet"]["kappa"] < 0.10:
    warn(f"<b>The U-Net did not converge</b> (kappa {res['UNet']['kappa']:.2f}) - it was probably stopped too early or hit the time cap. Do NOT interpret its scores; re-run, on a GPU runtime if possible.")

# ---- block-bootstrap 95% CIs: resample whole 2.56 km test blocks (paired across models) ----
rows_px, cols_px = te_idx // W, te_idx % W
blk = (rows_px // BK) * (W // BK) + cols_px // BK
cms_px = {"RandomForest": block_cms(y_true, pred_rf, blk), "UNet": block_cms(y_true, pred_unet, blk)}
n_test_blocks = cms_px["UNet"].shape[0]
boot = block_bootstrap(cms_px, CFG["boot_n"], CFG["seed"])
CI = {m: {k: ci(v) for k, v in d.items()} for m, d in boot.items()}          # (mean, lo, hi)
DIFF = {}                                                                    # U-Net minus RF: (observed, lo, hi)
for k in ("macro_f1", "kappa", "peat_f1"):
    d = boot["UNet"][k] - boot["RandomForest"][k]; d = d[np.isfinite(d)]
    a_, b_ = res["UNet"][k], res["RandomForest"][k]
    pt = (a_ - b_) if (a_ is not None and b_ is not None) else float("nan")
    DIFF[k] = (pt, float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))) if len(d) >= 20 else (pt, float("nan"), float("nan"))
if n_test_blocks < 10:
    warn(f"Only <b>{n_test_blocks}</b> test blocks: bootstrap intervals are wide and unstable, and resampling so few blocks understates the true uncertainty.")

# ---- 100 m aggregated scoring (CORINE's native resolution): modal class per KA x KA cell fully inside the test blocks ----
KA = CFG["agg_px"]
ref_agg, full = aggregate_modal(lab, TEST_MASK, KA)
agg_rf, _ = aggregate_modal(pred_rf_map, TEST_MASK, KA); agg_un, _ = aggregate_modal(pred_map, TEST_MASK, KA)
res_agg = {"RandomForest": score(ref_agg[full], agg_rf[full]), "UNet": score(ref_agg[full], agg_un[full])} if full.any() else None

# ---- save metrics ----
def _j(x): return None if x is None or (isinstance(x, float) and not np.isfinite(x)) else x
ci_json = {m: {k: [_j(x) for x in v[1:]] for k, v in d.items()} for m, d in CI.items()}      # [lo, hi]
diff_json = {k: [_j(x) for x in v] for k, v in DIFF.items()}                                 # [observed, lo, hi]
meta = dict(label_source=label_source, label_scheme=scheme, area=area["name"], test_pixels=int(len(y_true)), seed=CFG["seed"], split_seed=int(chosen_seed),
            unet_epochs_run=len(hist), unet_time_budget_hit=bool(budget_hit), mode=MODE, unet_converged=bool(res["UNet"]["kappa"] >= 0.10),
            unet_context_px=int(CFG["unet_context_px"]), n_test_blocks=int(n_test_blocks), bootstrap_resamples=int(CFG["boot_n"]),
            agg_cell_px=int(KA), agg_cells=int(full.sum()),
            note=("Labels are CORINE (100 m, 25 ha minimum mapping unit): scores are agreement with CORINE, not field-verified accuracy. CIs cover test-block sampling only. " if label_source == "corine"
                  else "PROXY labels: ESA WorldCover has no peat class (herbaceous wetland substituted). Not comparable to CORINE results."))
for k, v in res.items():
    json.dump({**v, "model": k, "ci95_block_bootstrap": ci_json[k], "aggregated_100m": (res_agg[k] if res_agg else None), **meta}, open(f"reports/metrics_{k}.json", "w"), indent=2)
json.dump({**meta, "models": res, "ci95_block_bootstrap": ci_json, "unet_minus_rf_ci95": diff_json, "aggregated_100m": res_agg,
           "config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()}}, open("reports/metrics.json", "w"), indent=2)

# ---- figures ----
def cm_norm(y, p):
    m = confusion_matrix(y, p, labels=LABS).astype(float); return m / np.maximum(m.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(1, 3, figsize=(20, 5.6), gridspec_kw=dict(width_ratios=[1, 1, 1.05]))
for a, (nm, p) in zip(ax[:2], [("Random Forest", pred_rf), ("U-Net", pred_unet)]):
    m = cm_norm(y_true, p); a.imshow(m, cmap="Blues", vmin=0, vmax=1)
    for i in LABS:
        for j in LABS: a.text(j, i, f"{m[i,j]:.2f}", ha="center", va="center", fontsize=8, color="white" if m[i, j] > .5 else "black")
    a.set_xticks(LABS); a.set_yticks(LABS); a.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=8); a.set_yticklabels(CLASS_NAMES, fontsize=8)
    a.set_xlabel("predicted"); a.set_ylabel("reference"); a.set_title(f"{nm} · row-normalised confusion", fontweight="bold")
wd = .38; xs = np.arange(7)
ax[2].bar(xs - wd/2, [res["RandomForest"]["per_class"][n]["f1"] for n in CLASS_NAMES], wd, label="Random Forest", color="#2a9d8f")
ax[2].bar(xs + wd/2, [res["UNet"]["per_class"][n]["f1"] for n in CLASS_NAMES], wd, label="U-Net", color="#e76f51")
ax[2].set_xticks(xs); ax[2].set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=8); ax[2].set_ylim(0, 1.18); ax[2].set_ylabel("F1")
ax[2].axvspan(PEAT - .5, PEAT + .5, color="#6d4c41", alpha=.12); ax[2].set_title("Per-class F1 (peat highlighted)", fontweight="bold"); ax[2].legend(); ax[2].grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig("reports/04_results.png", dpi=130); plt.show()

f = lambda v: "n/a" if v is None else f"{v:.3f}"
def fci(m, k):
    v = res[m][k]; _, lo, hi = CI[m][k]
    return "n/a" if v is None else (f"{v:.3f}" if not np.isfinite(lo) else f"{v:.3f} [{lo:.3f}–{hi:.3f}]")
best_name = max(res, key=lambda k_: (res[k_]["peat_f1"] or -1))
html_table([[k_, f"{v['accuracy']:.3f}", fci(k_, "macro_f1"), fci(k_, "kappa"), fci(k_, "peat_f1"), f(v["peat_precision"]), f(v["peat_recall"])] for k_, v in res.items()],
           ["model", "overall accuracy", "macro F1 [95% CI]", "kappa [95% CI]", "🟤 peat F1 [95% CI]", "peat precision", "peat recall"],
           highlight=list(res).index(best_name), width="95%")
d_m, d_p = DIFF["macro_f1"], DIFF["peat_f1"]
info(f"<b>U-Net minus RF (paired, {n_test_blocks} test blocks):</b> macro F1 {d_m[0]:+.3f} [{d_m[1]:+.3f}, {d_m[2]:+.3f}] · peat F1 {d_p[0]:+.3f} [{d_p[1]:+.3f}, {d_p[2]:+.3f}]. "
     "An interval that contains 0 means the data cannot separate the two models.")
if res_agg:
    html_table([[k_, f"{v['accuracy']:.3f}", f"{v['macro_f1']:.3f}", f"{v['kappa']:.3f}", f(v["peat_f1"]), f(v["peat_precision"]), f(v["peat_recall"])] for k_, v in res_agg.items()],
               ["model (100 m aggregated)", "overall accuracy", "macro F1", "kappa", "🟤 peat F1", "peat precision", "peat recall"], width="85%")
    info(f"100 m scoring: modal class per {KA}x{KA}-pixel cell, {int(full.sum()):,} cells fully inside test blocks. This is closer to CORINE's native resolution than the 10 m pixel scores.")

# ---- figure: test maps · Reference · RF · U-Net · peat errors (journal style; legends live in a dedicated side column, never on the maps) ----
ref_map = np.where(TEST_MASK, lab, 255); s = PS
RES_M = CFG["res"]                                                        # metres per raster pixel (10 m)
NX_KM = W * RES_M / 1000.0                                                # true map width in km

def _frame(a, title):
    a.set_title(title, fontweight="bold", fontsize=11.5, pad=8)
    a.set_xticks([]); a.set_yticks([])                                    # framed panel (neatline)
    for sp in a.spines.values(): sp.set_linewidth(0.9); sp.set_color("#1a1a1a")

def show(a, m, title):
    a.imshow(np.where(m == 255, 7, m)[::s, ::s], cmap=CMAP, vmin=0, vmax=7, interpolation="nearest")
    _frame(a, title)

fig = plt.figure(figsize=(14.6, 12.4), constrained_layout=True)
gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 0.40])                    # 2x2 maps + legend column
ax = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]),
      fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])]
axleg = fig.add_subplot(gs[:, 2]); axleg.set_xticks([]); axleg.set_yticks([])
for sp in axleg.spines.values(): sp.set_visible(False)

show(ax[0], ref_map, "(a) Reference (test blocks only)")
show(ax[1], np.where(TEST_MASK, pred_rf_map, 255), "(b) Random Forest")
show(ax[2], np.where(TEST_MASK, pred_map, 255), "(c) U-Net")
bm = pred_rf_map if best_name == "RandomForest" else pred_map
agree = np.zeros((H, W), np.uint8); agree[TEST_MASK] = 1
agree[TEST_MASK & (lab == PEAT) & (bm == PEAT)] = 2; agree[TEST_MASK & (lab == PEAT) & (bm != PEAT)] = 3; agree[TEST_MASK & (lab != PEAT) & (bm == PEAT)] = 4
ax[3].imshow(agree[::s, ::s], cmap=ListedColormap(["#ffffff", "#e5e7eb", "#16a34a", "#dc2626", "#f59e0b"]), vmin=0, vmax=4, interpolation="nearest")
_frame(ax[3], f"(d) Peat bogs · {best_name}")                            # frame+title only; imshow above sets the colours

# ---- dedicated legend column: N arrow · class legend · error legend · scale bar
axleg.annotate("N", xy=(0.5, 0.965), xytext=(0.5, 0.905),                 # delete if raster is not north-up
               arrowprops=dict(arrowstyle="-|>", mutation_scale=14, lw=1.1, color="#1a1a1a"),
               ha="center", va="bottom", fontsize=12, fontweight="bold", color="#1a1a1a")
cls_h = [patches.Patch(color=CLASS_COLORS[i], label=CLASS_NAMES[i]) for i in range(7)] + \
        [patches.Patch(facecolor="#ffffff", edgecolor="#bbbbbb", label="No data / outside")]
err_h = [patches.Patch(color=c, label=l) for c, l in
         zip(["#16a34a", "#dc2626", "#f59e0b"], ["correct peat", "missed peat", "false alarm"])] + \
        [patches.Patch(color="#e5e7eb", label="non-peat (reference)"),
         patches.Patch(facecolor="#ffffff", edgecolor="#bbbbbb", label="outside test blocks")]
lstyle = dict(fontsize=9.5, framealpha=.95, edgecolor="#bbbbbb", labelspacing=.7,
              handlelength=1.5, handleheight=1.2)
leg1 = axleg.legend(handles=cls_h, loc="upper center", bbox_to_anchor=(0.5, 0.865),
                    title=f"Land cover class ({RES_M:.0f} m)", title_fontsize=10.5, **lstyle)
axleg.add_artist(leg1)
leg2 = axleg.legend(handles=err_h, loc="upper center", bbox_to_anchor=(0.5, 0.375),
                    title="Peat error map", title_fontsize=10.5, **lstyle)

# scale bar: physically calibrated against panel (a)'s printed width -> honest length
fig.canvas.draw(); _r = fig.canvas.get_renderer()
_mw = ax[0].get_window_extent(_r).width / fig.dpi                         # map panel width (inches)
_aw = axleg.get_window_extent(_r).width / fig.dpi                         # legend column width (inches)
_bar_km = next((v for v in (100, 50, 20, 10, 5, 2, 1) if v <= 0.35 * NX_KM), 0.5)
_w = (_bar_km / NX_KM) * _mw / _aw                                        # bar as fraction of column
axleg.add_patch(patches.Rectangle((0.5 - _w / 2, 0.075), _w, 0.016, transform=axleg.transAxes,
                                  facecolor="#1a1a1a", edgecolor="none"))
axleg.text(0.5, 0.103, f"{_bar_km} km", transform=axleg.transAxes, ha="center", va="bottom", fontsize=10)

plt.savefig("reports/05_test_maps.png", dpi=200, bbox_inches="tight"); plt.show()

# ---------------------------------------------------------------------------
# WIDE 1x4 BANNER VARIANT (for slides) — swap the whole block above for:
#   fig, ax = plt.subplots(1, 4, figsize=(22, 6.4), constrained_layout=True)
#   ... identical show(...) lines for panels (a)-(c), identical agree lines,
#       then ax[3].imshow(...); _frame(ax[3], f"(d) Peat bogs · {best_name}") ...
#   ax[0].legend(handles=cls_h, loc="upper center", bbox_to_anchor=(0.5, -0.03),
#                ncol=2, fontsize=9.5, framealpha=.95, edgecolor="#bbbbbb",
#                labelspacing=.55, columnspacing=1.3, handlelength=1.4)
#   ax[3].legend(handles=err_h, loc="upper center", bbox_to_anchor=(0.5, -0.03),
#                ncol=2, fontsize=9.5, framealpha=.95, edgecolor="#bbbbbb",
#                labelspacing=.55, columnspacing=1.3, handlelength=1.4)
#   plt.savefig("reports/05_test_maps.png", dpi=200, bbox_inches="tight"); plt.show()
# (legends sit in the margin below panels (a) and (d); bbox_inches="tight"
#  grows the canvas, so they can never be clipped or overlap the maps)
# ---------------------------------------------------------------------------


## Block 11b · Uncertainty estimation (MC dropout)
**What it does:** estimates how confident the trained U-Net is, pixel by pixel, using **Monte Carlo (MC) dropout**. A `Dropout2d(p=0.15)` layer is inserted before the segmentation head of a **deep copy** of the trained model, and the copy is fine-tuned briefly (low learning rate, time-boxed, best state chosen on validation loss) so it adapts to the dropout noise - the original model and every Block 11 number stay untouched. At prediction time the dropout layers stay active while BatchNorm stays frozen, and `T` stochastic passes (default 20, `CFG["mc_T"]`; automatically reduced if the Block 11b time budget would be exceeded) produce, per pixel: the **mean softmax probability**, the **predicted class** (argmax of the mean) and the **predictive entropy** as the uncertainty map. Nodata (255) pixels are ignored everywhere.

**What is checked (same unseen test pixels as Block 11):** mean uncertainty for correctly vs incorrectly classified pixels; accuracy and peat F1 after rejecting the most uncertain 10%, 20% and 30% of pixels; a reliability diagram and the Expected Calibration Error (15 bins) for all classes and for peat (one-vs-rest). One figure shows the uncertainty map next to the prediction, the labels and the reliability diagram, in the style of the earlier figures. An optional 3-seed deep ensemble (`CFG["do_ensemble"]`, default `False`) provides a comparison. Everything is saved to `reports/uncertainty.json` and `reports/uncertainty.png`; Block 13 zips the whole `reports/` folder, so both files are exported automatically.

**Limitations (read before quoting any number):** MC dropout is an *approximation* of Bayesian uncertainty, and the only stochastic layer added here sits in front of the segmentation head, so the map highlights where the final decision is unstable - it cannot flag errors the features themselves cannot see. Uncertainty is measured *against CORINE labels* (100 m grid, 25 ha minimum mapping unit), so part of what it marks as "uncertain" is label coarseness, not model weakness. Calibration and rejection results come from one area, one season and one block split; a rejected pixel is not guaranteed wrong, and an accepted pixel is not guaranteed right.

Run the four code cells in order; they only need Blocks 1-11 (trained U-Net, test tiles, labels) to have run.

In [ ]:
banner("11b", "Uncertainty estimation (MC dropout) · dropout & fine-tune", "a deep copy of the trained U-Net gets a Dropout2d before its segmentation head · brief time-boxed fine-tune · no existing result is touched")
UNC_T0 = time.time()
# ---- Block 11b settings (defined HERE so Blocks 1-11 keep producing exactly the same results as before) ----
CFG.update(mc_T=20, mc_dropout_p=0.15, mc_ft_epochs=3, mc_ft_lr_scale=0.25, mc_ece_bins=15,
           unc_budget_s=int(0.50 * CFG["unet_budget_s"]),       # total extra budget: fine-tune + MC passes
           unc_ft_budget_s=int(0.35 * CFG["unet_budget_s"]),    # share reserved for the fine-tune
           unc_ens_budget_s=max(60, CFG["unet_budget_s"] // 3), do_ensemble=False)
html_table([[k, str(v)] for k, v in CFG.items() if k.startswith(("mc_", "unc_")) or k == "do_ensemble"],
           ["Block 11b setting", "value"], width="55%")
UNC_FINETUNED, hist_mc, best_mc = False, [], float("inf")   # inf = "no best yet", same convention as Block 10
torch.manual_seed(CFG["seed"])                       # same seed handling as Block 1
model_mc = copy.deepcopy(model)                      # Block 10's `model` is never modified
model_mc.segmentation_head = nn.Sequential(nn.Dropout2d(p=float(CFG["mc_dropout_p"])), *list(model_mc.segmentation_head.children()))
ok(f"<b>Option chosen:</b> insert <code>Dropout2d(p={CFG['mc_dropout_p']})</code> before the segmentation head of a <b>deep copy</b> of the trained U-Net and fine-tune briefly. "
   "<b>Why:</b> this is the smallest change that works - the copy keeps every learned feature and only adapts to the dropout noise it will face at inference, while training a fresh "
   "dropout variant from scratch would spend most of the time budget re-learning the same task. Working on a copy means no existing cell's results can change.")
if res["UNet"]["kappa"] < 0.10:
    warn("The U-Net did not converge in Block 10 (kappa < 0.10) - uncertainty numbers below describe an unreliable model and should not be quoted.")
unc_ft_t0 = time.time()
if Xtr is None:
    unc_ft_best_state = copy.deepcopy(model_mc.state_dict())
    warn("No training tiles found - MC dropout is applied WITHOUT the fine-tune; treat the probabilities as uncalibrated.")
else:
    UNC_FINETUNED = True
    cnt_u = np.bincount(Ytr[Ytr != 255].numpy().ravel(), minlength=7).astype(np.float64)
    w_u = np.where(cnt_u > 0, (cnt_u.sum() / (7 * np.maximum(cnt_u, 1))) ** 0.5, 0.0); w_u = w_u / w_u[cnt_u > 0].mean()
    crit_u = nn.CrossEntropyLoss(weight=torch.tensor(w_u, dtype=torch.float32, device=DEVICE), ignore_index=255)
    opt_u = torch.optim.AdamW(model_mc.parameters(), lr=CFG["unet_lr"] * CFG["mc_ft_lr_scale"], weight_decay=1e-4)
    try: scaler_u = torch.amp.GradScaler("cuda", enabled=use_amp)
    except Exception: scaler_u = torch.cuda.amp.GradScaler(enabled=use_amp)
    g_u = torch.Generator().manual_seed(CFG["seed"]); arng_u = np.random.default_rng(CFG["seed"])
    def augment_u(xb, yb):
        k = int(arng_u.integers(4)); xb, yb = torch.rot90(xb, k, (2, 3)), torch.rot90(yb, k, (1, 2))
        if arng_u.random() < .5: xb, yb = torch.flip(xb, (3,)), torch.flip(yb, (2,))
        return xb, yb
    unc_ft_best_state = None
    for ep in range(1, CFG["mc_ft_epochs"] + 1):
        model_mc.train(); perm = torch.randperm(len(Xtr), generator=g_u); tl, nb_u = 0.0, 0
        for i in range(0, len(perm), BS):
            ib = perm[i:i + BS]; xb, yb = augment_u(Xtr[ib], Ytr[ib]); xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt_u.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp): loss = crit_u(model_mc(xb), yb)
            scaler_u.scale(loss).backward(); scaler_u.step(opt_u); scaler_u.update(); tl += float(loss); nb_u += 1
            if time.time() - unc_ft_t0 > CFG["unc_ft_budget_s"]: break
        model_mc.eval(); vl, vn_u = 0.0, 0
        with torch.no_grad():
            for i in range(0, len(Xva), 32):
                xb, yb = Xva[i:i + 32].to(DEVICE), Yva[i:i + 32].to(DEVICE)
                with torch.autocast(device_type=DEVICE.type, enabled=use_amp): vl += float(crit_u(model_mc(xb), yb)) * len(xb)
                vn_u += len(xb)
        tl, vl = tl / max(nb_u, 1), vl / max(vn_u, 1); hist_mc.append((tl, vl))
        star = ""
        if vl < best_mc - 1e-4: best_mc, unc_ft_best_state, star = vl, copy.deepcopy(model_mc.state_dict()), " ⭐ best"
        print(f"fine-tune epoch {ep}/{CFG['mc_ft_epochs']} | train {tl:.3f} | val {vl:.3f} | {time.time()-unc_ft_t0:5.0f}s{star}")
        if time.time() - unc_ft_t0 > CFG["unc_ft_budget_s"]: print("fine-tune time box reached - continuing with the best state so far"); break
    if unc_ft_best_state is None: unc_ft_best_state = copy.deepcopy(model_mc.state_dict())
model_mc.load_state_dict(unc_ft_best_state)
UNC_FT_S = time.time() - unc_ft_t0
ok(f"Dropout model ready: fine-tuned {len(hist_mc)} epoch(s) in {UNC_FT_S:.0f} s (box {CFG['unc_ft_budget_s']} s)"
   + (f" · best fine-tune validation loss {best_mc:.3f}" if best_mc is not None and np.isfinite(best_mc) else " · no fine-tune (validation loss n/a)"))
info("Next cell runs the stochastic MC passes over the test tiles: dropout active, BatchNorm frozen, same context windows as Block 10.")

In [ ]:
banner("11b", "MC dropout inference", "dropout layers active · BatchNorm frozen · T stochastic passes over the test tiles (context windows as in Block 10) · nodata ignored")
te_tiles_u = np.argwhere(TS == 2)
if len(te_tiles_u) == 0: raise RuntimeError("No test tiles found - run Blocks 1-11 first (Block 11b needs the trained U-Net and the test tiles).")
prob_sum = np.zeros((7, H, W), np.float32)                     # running sum of the softmax over passes
model_mc.eval()                                                # BatchNorm (and everything else) frozen...
for _m in model_mc.modules():
    if isinstance(_m, (nn.Dropout, nn.Dropout2d, nn.Dropout3d)): _m.train()   # ...only dropout stays stochastic
def _mc_pass():
    # One stochastic forward sweep over all test tiles; adds the softmax to prob_sum.
    for i in range(0, len(te_tiles_u), 16):
        chunk = te_tiles_u[i:i + 16]
        xb = torch.from_numpy(np.stack([_window(r, c) for r, c in chunk])).to(DEVICE)
        # autocast here (Block 10's single test pass is fp32) keeps the T passes fast; the precision
        # difference is ~1e-3 in probability and does not change the comparison materially
        with torch.autocast(device_type=DEVICE.type, enabled=use_amp): out = model_mc(xb)
        sm = torch.softmax(out.float(), dim=1).cpu().numpy().astype(np.float32)[:, :, P:P + T, P:P + T]
        for (r, c), p in zip(chunk, sm): prob_sum[:, r*T:(r+1)*T, c*T:(c+1)*T] += p
T_EFF, UNC_PASS_S = int(CFG["mc_T"]), 0.0
with torch.no_grad():
    torch.manual_seed(CFG["seed"])                             # reproducible dropout masks from the existing seed setup
    t0p = time.time(); _mc_pass(); UNC_PASS_S = time.time() - t0p
    budget_left = CFG["unc_budget_s"] - UNC_FT_S - UNC_PASS_S
    if CFG["mc_T"] > 1 and budget_left < (CFG["mc_T"] - 1) * UNC_PASS_S:
        # v10: guard against a negative remaining budget - always run at least the minimum 2 passes,
        # explicitly flagged when the budget was already spent by the fine-tune + first pass
        T_EFF = max(2, int(max(budget_left, 0.0) // UNC_PASS_S) + 1)
        if T_EFF < CFG["mc_T"]:
            _extra = " The budget was already spent by the fine-tune and the first pass - running the minimum 2 passes and accepting a small overrun." if budget_left <= 0 else ""
            warn(f"<b>T reduced from {CFG['mc_T']} to {T_EFF} passes</b> to stay inside the Block 11b time budget "
                 f"({UNC_PASS_S:.1f} s per measured pass, {CFG['unc_budget_s']} s budget). More passes would give smoother uncertainty estimates.{_extra}")
    t0p = time.time()
    for _ in range(1, T_EFF): _mc_pass()
    UNC_MC_S = time.time() - t0p
prob_sum /= np.float32(T_EFF); mean_prob = prob_sum; prob_sum = None   # mean predictive distribution per pixel
unc_pred_map = mean_prob.argmax(0).astype(np.uint8)                    # predicted class from the MEAN softmax
UNC_ENTROPY = (-(mean_prob * np.log(np.maximum(mean_prob, 1e-12))).sum(0)).astype(np.float32)   # predictive entropy (nats, 0..ln 7)
UNC_CONF = mean_prob.max(0).astype(np.float32)                         # confidence of the mean distribution
UNC_PPEAT = mean_prob[PEAT].astype(np.float32).copy()                  # mean probability of class 4 (peat)
mean_prob = None                                                       # free the big array - only reduced maps are kept
UNC_MC_TOT_S = UNC_PASS_S + UNC_MC_S
ok(f"MC dropout done: {T_EFF} passes over {len(te_tiles_u)} test tiles · {UNC_MC_TOT_S:.0f} s inference ({UNC_PASS_S:.1f} s/pass) · "
   f"entropy on test pixels ranges {UNC_ENTROPY[TEST_MASK].min():.3f}-{UNC_ENTROPY[TEST_MASK].max():.3f} nats (maximum possible ln 7 = {np.log(7):.3f})")

In [ ]:
banner("11b", "Uncertainty checks · reliability · rejection", "same test pixels as Block 11 · every number computed in this run")
def ece_bins(conf, outcome, bins):
    # Equal-width confidence bins -> (ECE, mean confidence per bin, accuracy per bin, counts); empty bins carry no weight.
    conf = np.asarray(conf, np.float64); outcome = np.asarray(outcome, bool)
    ids = np.minimum((conf * bins).astype(int), bins - 1)
    n_b = np.bincount(ids, minlength=bins).astype(np.float64)
    acc_b = np.divide(np.bincount(ids, weights=outcome.astype(np.float64), minlength=bins), np.maximum(n_b, 1))
    cf_b = np.divide(np.bincount(ids, weights=conf, minlength=bins), np.maximum(n_b, 1))
    return float((n_b * np.abs(acc_b - cf_b)).sum() / max(n_b.sum(), 1)), cf_b, acc_b, n_b
fmt3 = lambda v: "n/a" if v is None or not np.isfinite(v) else f"{v:.3f}"
pred_mc = unc_pred_map.ravel()[te_idx]                 # identical test pixels as pred_unet in Block 11
_mc_sc = score(y_true, pred_mc)
info(f"<b>The MC-dropout model is a fine-tuned copy, not the Block-10 U-Net.</b> Test accuracy {_mc_sc['accuracy']:.3f} (Block 10: {res['UNet']['accuracy']:.3f}) · "
     f"macro F1 {_mc_sc['macro_f1']:.3f} (Block 10: {res['UNet']['macro_f1']:.3f}) · peat F1 {fmt3(_mc_sc['peat_f1'])} (Block 10: {fmt3(res['UNet']['peat_f1'])}). "
     "Uncertainty numbers below describe the dropout model.")
UNC_CORRECT = pred_mc == y_true
ent_flat = UNC_ENTROPY.ravel()[te_idx]

# ---- 1 · mean uncertainty for correct vs incorrect pixels ----
n_ok, n_bad = int(UNC_CORRECT.sum()), int((~UNC_CORRECT).sum())
e_ok = float(ent_flat[UNC_CORRECT].mean()) if n_ok else float("nan")
e_bad = float(ent_flat[~UNC_CORRECT].mean()) if n_bad else float("nan")
html_table([["correctly classified", f"{n_ok:,}", fmt3(e_ok)], ["misclassified", f"{n_bad:,}", fmt3(e_bad)]],
           ["test pixels", "count", "mean predictive entropy (nats)"], width="60%")
info("Entropy clearly higher for misclassified pixels means the uncertainty map carries signal; near-equal values mean it does not separate them.")

# ---- 2 · accuracy and peat F1 after rejecting the most uncertain 10/20/30% ----
order = np.argsort(ent_flat)[::-1]                     # most uncertain test pixels first
n_t = len(ent_flat); UNC_REJ = []
for r_pct in (0, 10, 20, 30):
    n_drop = int(round(n_t * r_pct / 100)); keep = order[n_drop:]
    mm = score(y_true[keep], pred_mc[keep])
    UNC_REJ.append(dict(reject_pct=r_pct, retained_px=int(len(keep)), accuracy=float(mm["accuracy"]),
                        peat_f1=(None if mm["peat_f1"] is None else float(mm["peat_f1"])),
                        mean_entropy_kept=float(ent_flat[keep].mean())))
html_table([[f"{d['reject_pct']}%", f"{d['retained_px']:,}", f"{d['accuracy']:.3f}", fmt3(d['peat_f1']), f"{d['mean_entropy_kept']:.3f}"] for d in UNC_REJ],
           ["most uncertain pixels rejected", "retained test px", "accuracy", "🟤 peat F1", "mean entropy of kept px"], width="70%")
info("Useful uncertainty RAISES accuracy and peat F1 as coverage is given up. A rejected pixel is not guaranteed wrong - coverage is traded for purity.")

# ---- 3 · reliability diagram + ECE (15 bins), all classes and peat ----
ECE_ALL, cf_a, acc_a, n_a = ece_bins(UNC_CONF.ravel()[te_idx], UNC_CORRECT, CFG["mc_ece_bins"])
ECE_PEAT, cf_p, acc_p, n_p = ece_bins(UNC_PPEAT.ravel()[te_idx], y_true == PEAT, CFG["mc_ece_bins"])
html_table([["all classes (max-probability confidence)", f"{ECE_ALL:.3f}"], ["peat bogs, one-vs-rest (probability of class 4)", f"{ECE_PEAT:.3f}"]],
           ["reliability", f"ECE ({CFG['mc_ece_bins']} bins)"], width="55%")

# ---- figure: uncertainty map · prediction · labels · reliability diagram ----
s = PS  # display stride (same alias Block 11 sets; kept here so this cell is robust on its own)
def show(a, m, title):                                                    # framed version, self-contained
    a.imshow(np.where(m == 255, 7, m)[::s, ::s], cmap=CMAP, vmin=0, vmax=7, interpolation="nearest")
    a.set_title(title, fontweight="bold", fontsize=11, pad=8)
    a.set_xticks([]); a.set_yticks([])
    for sp in a.spines.values(): sp.set_linewidth(0.9); sp.set_color("#1a1a1a")

ent_fig = np.where(TEST_MASK, UNC_ENTROPY, np.nan)
cmap_u = plt.get_cmap("magma").copy(); cmap_u.set_bad("#ffffff")
fig, axs = plt.subplots(2, 2, figsize=(15.5, 12.0), constrained_layout=True)
im = axs[0, 0].imshow(ent_fig[::PS, ::PS], cmap=cmap_u, vmin=0, vmax=float(np.log(7)), interpolation="nearest")
axs[0, 0].set_title(f"(a) Predictive entropy · MC dropout, T={T_EFF} (white = outside test blocks / nodata)",
                    fontweight="bold", fontsize=11, pad=8)
axs[0, 0].set_xticks([]); axs[0, 0].set_yticks([])
for sp in axs[0, 0].spines.values(): sp.set_linewidth(0.9); sp.set_color("#1a1a1a")
plt.colorbar(im, ax=axs[0, 0], fraction=0.046, pad=0.02).set_label("entropy (nats)")
show(axs[0, 1], np.where(TEST_MASK, unc_pred_map, 255), "(b) Prediction (argmax of mean softmax)")
show(axs[1, 0], np.where(TEST_MASK, lab, 255), "(c) Reference labels")
axr = axs[1, 1]
axr.plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
ma, mp_ = n_a > 0, n_p > 0
axr.plot(cf_a[ma], acc_a[ma], "o-", color="#e76f51", label=f"all classes · ECE {ECE_ALL:.3f}")
axr.plot(cf_p[mp_], acc_p[mp_], "s-", color="#2a9d8f", label=f"peat (one-vs-rest) · ECE {ECE_PEAT:.3f}")
axr.set_xlim(0, 1); axr.set_ylim(0, 1); axr.set_xlabel("mean predicted probability"); axr.set_ylabel("observed frequency")
axr.set_title(f"(d) Reliability diagram ({CFG['mc_ece_bins']} bins)", fontweight="bold", fontsize=11, pad=8)
axr.legend(loc="upper left", fontsize=9)                                  # empty quadrant, no curve collision
axr.grid(alpha=.3)

# class legend: dedicated framed row BELOW the whole grid (touches nothing)
cls_h = [patches.Patch(color=CLASS_COLORS[i], label=CLASS_NAMES[i]) for i in range(7)] + \
        [patches.Patch(facecolor="#ffffff", edgecolor="#bbbbbb", label="No data / outside")]
fig.legend(handles=cls_h, loc="outside lower center", ncol=4, fontsize=9.5,
           frameon=True, framealpha=.95, edgecolor="#bbbbbb", borderaxespad=0.5,
           columnspacing=1.6, handlelength=1.5, handleheight=1.1, labelspacing=.55)
# (matplotlib >= 3.7; if older, delete the fig.legend(...) above and use:
#   axs[1, 0].legend(handles=cls_h, loc="upper center", bbox_to_anchor=(0.5, -0.04),
#                    ncol=4, fontsize=9, framealpha=.95, edgecolor="#bbbbbb"))

plt.savefig("reports/uncertainty.png", dpi=200, bbox_inches="tight"); plt.show()
# ---- save ----
def _jnum(x):
    if x is None: return None
    if isinstance(x, (bool, np.bool_)): return bool(x)
    if isinstance(x, (int, np.integer)): return int(x)
    if isinstance(x, (float, np.floating)): return float(x) if np.isfinite(x) else None
    return x
UNC_RUNTIME_S = time.time() - UNC_T0
json.dump(dict(
    block="11b · Uncertainty estimation (MC dropout)", method="MC dropout",
    settings=dict(mc_T_requested=int(CFG["mc_T"]), mc_T_run=int(T_EFF), dropout_p=float(CFG["mc_dropout_p"]),
                  dropout_location="Dropout2d before the segmentation head of a deep copy of the trained U-Net",
                  fine_tuned=bool(UNC_FINETUNED), fine_tune_epochs_run=int(len(hist_mc)), fine_tune_val_loss_best=_jnum(best_mc),
                  ece_bins=int(CFG["mc_ece_bins"]), uncertainty_budget_s=int(CFG["unc_budget_s"]), seed=int(CFG["seed"]), mode=MODE),
    runtime_s=dict(fine_tune=round(float(UNC_FT_S), 1), mc_inference=round(float(UNC_MC_TOT_S), 1),
                   seconds_per_mc_pass=round(float(UNC_PASS_S), 2), block_total_so_far=round(float(UNC_RUNTIME_S), 1)),
    ece=dict(all_classes=round(ECE_ALL, 4), peat_one_vs_rest=round(ECE_PEAT, 4)),
    entropy_correct_vs_incorrect=dict(correct_px=n_ok, correct_mean_entropy=_jnum(e_ok),
                                      misclassified_px=n_bad, misclassified_mean_entropy=_jnum(e_bad)),
    rejection=UNC_REJ, test_pixels=int(len(y_true)), label_source=label_source,
    note=("MC dropout approximates Bayesian uncertainty by keeping dropout active at inference; the only stochastic layer sits before the segmentation head. "
          "Uncertainty is measured against CORINE labels (100 m, 25 ha minimum mapping unit), so part of it reflects label coarseness rather than model weakness. "
          "Rejection and calibration come from one area, one season and one block split; a rejected pixel is not guaranteed wrong.")),
    open("reports/uncertainty.json", "w"), indent=2)
ok("Saved <code>reports/uncertainty.json</code> · <code>reports/uncertainty.png</code> - Block 13 zips the whole reports/ folder, so both are exported automatically")

In [ ]:
banner("11b", "Optional deep ensemble & how to use this block", "3-seed ensemble only if CFG[\"do_ensemble\"] is True · then: cell order, measured extra runtime, honest CV line")
ENS_RAN, UNC_ENS_S = False, 0.0
if not CFG["do_ensemble"]:
    info('Optional deep ensemble <b>skipped</b> - <code>CFG["do_ensemble"]</code> is False (default). Set it to True in the first Block 11b cell and re-run the block to compare MC dropout with a 3-seed ensemble.')
elif Xtr is None:
    warn("Deep ensemble skipped - no training tiles are available.")
else:
    t_ens = time.time(); prob_sum_e = np.zeros((7, H, W), np.float32); ens_epochs = []
    for m_i in range(3):
        seed_m = CFG["seed"] + 1001 + m_i
        torch.manual_seed(seed_m); rng_m = np.random.default_rng(seed_m)
        m_m = smp.Unet("resnet18", encoder_weights=None, in_channels=len(FEATURES), classes=7,
                       decoder_channels=(128, 64, 32, 16, 16)).to(DEVICE)
        opt_m = torch.optim.AdamW(m_m.parameters(), lr=CFG["unet_lr"], weight_decay=1e-4)
        try: scaler_m = torch.amp.GradScaler("cuda", enabled=use_amp)
        except Exception: scaler_m = torch.cuda.amp.GradScaler(enabled=use_amp)
        g_m = torch.Generator().manual_seed(seed_m); t_m = time.time(); best_m, best_state_m, hist_m = float("inf"), None, 0
        for ep in range(1, CFG["mc_ft_epochs"] + 1):
            m_m.train(); perm = torch.randperm(len(Xtr), generator=g_m); tl, nb_m = 0.0, 0
            for i in range(0, len(perm), BS):
                ib = perm[i:i + BS]
                k = int(rng_m.integers(4)); xb, yb = torch.rot90(Xtr[ib], k, (2, 3)), torch.rot90(Ytr[ib], k, (1, 2))
                if rng_m.random() < .5: xb, yb = torch.flip(xb, (3,)), torch.flip(yb, (2,))
                xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt_m.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, enabled=use_amp): loss = crit(m_m(xb), yb)
                scaler_m.scale(loss).backward(); scaler_m.step(opt_m); scaler_m.update(); tl += float(loss); nb_m += 1
                if time.time() - t_m > CFG["unc_ens_budget_s"]: break
            m_m.eval(); vl, vn_m = 0.0, 0
            with torch.no_grad():
                for i in range(0, len(Xva), 32):
                    xb, yb = Xva[i:i + 32].to(DEVICE), Yva[i:i + 32].to(DEVICE)
                    with torch.autocast(device_type=DEVICE.type, enabled=use_amp): vl += float(crit(m_m(xb), yb)) * len(xb)
                    vn_m += len(xb)
            vl = vl / max(vn_m, 1); hist_m += 1
            if vl < best_m - 1e-4: best_m, best_state_m = vl, copy.deepcopy(m_m.state_dict())
            print(f"ensemble member {m_i+1}/3 (seed {seed_m}) epoch {ep} | train {tl/max(nb_m,1):.3f} | val {vl:.3f} | {time.time()-t_m:4.0f}s")
            if time.time() - t_m > CFG["unc_ens_budget_s"]: print("  member time box reached"); break
        if best_state_m is not None: m_m.load_state_dict(best_state_m)
        m_m.eval()
        with torch.no_grad():
            for i in range(0, len(te_tiles_u), 16):
                chunk = te_tiles_u[i:i + 16]
                xb = torch.from_numpy(np.stack([_window(r, c) for r, c in chunk])).to(DEVICE)
                with torch.autocast(device_type=DEVICE.type, enabled=use_amp): out = m_m(xb)
                sm = torch.softmax(out.float(), dim=1).cpu().numpy().astype(np.float32)[:, :, P:P + T, P:P + T]
                for (r, c), p in zip(chunk, sm): prob_sum_e[:, r*T:(r+1)*T, c*T:(c+1)*T] += p
        ens_epochs.append(hist_m); del m_m, opt_m, scaler_m
    mean_prob_e = prob_sum_e / 3.0; prob_sum_e = None
    unc_pred_e = mean_prob_e.argmax(0).astype(np.uint8).ravel()[te_idx]
    ENT_E = (-(mean_prob_e * np.log(np.maximum(mean_prob_e, 1e-12))).sum(0)).astype(np.float32)
    CONF_E = mean_prob_e.max(0).astype(np.float32); PPEAT_E = mean_prob_e[PEAT].astype(np.float32).copy(); mean_prob_e = None
    m_mc, m_en = score(y_true, pred_mc), score(y_true, unc_pred_e)
    ECE_ALL_E, _, _, _ = ece_bins(CONF_E.ravel()[te_idx], unc_pred_e == y_true, CFG["mc_ece_bins"])
    ECE_PEAT_E, _, _, _ = ece_bins(PPEAT_E.ravel()[te_idx], y_true == PEAT, CFG["mc_ece_bins"])
    _r = float(np.corrcoef(ent_flat, ENT_E.ravel()[te_idx])[0, 1])
    html_table([["MC dropout", f"T={T_EFF} stochastic passes", f"{m_mc['accuracy']:.3f}", fmt3(m_mc["peat_f1"]), f"{ECE_ALL:.3f}", f"{ECE_PEAT:.3f}", f"{ent_flat.mean():.3f}"],
                ["Deep ensemble", "3 seeds (short runs)", f"{m_en['accuracy']:.3f}", fmt3(m_en["peat_f1"]), f"{ECE_ALL_E:.3f}", f"{ECE_PEAT_E:.3f}", f"{ENT_E.ravel()[te_idx].mean():.3f}"]],
               ["uncertainty source", "setup", "accuracy", "🟤 peat F1", "ECE (all)", "ECE (peat)", "mean entropy"], width="90%")
    info(f"Pearson r between the two uncertainty maps on test pixels: {_r:.3f} (1 = they rank the same pixels as uncertain). Ensemble members ran only "
         f"{ens_epochs} epoch(s) each, so treat the ensemble row as an indicative comparison, not a fully trained alternative.")
    ENS_RAN = True; UNC_ENS_S = time.time() - t_ens
    J = json.load(open("reports/uncertainty.json")); J["settings"]["ensemble_run"] = True
    J["ensemble"] = dict(run=True, members=3, seeds=[int(CFG["seed"]) + 1001, int(CFG["seed"]) + 1002, int(CFG["seed"]) + 1003],
                         epochs_run=[int(e) for e in ens_epochs], runtime_s=round(time.time() - t_ens, 1),
                         accuracy=float(m_en["accuracy"]), peat_f1=_jnum(m_en["peat_f1"]),
                         ece_all_classes=round(ECE_ALL_E, 4), ece_peat_one_vs_rest=round(ECE_PEAT_E, 4),
                         mean_entropy_test_px=float(ENT_E.ravel()[te_idx].mean()),
                         entropy_pearson_r_vs_mc=_jnum(_r),
                         note="Members are short, time-boxed runs - the ensemble is a comparison point for MC dropout, not a fully trained alternative.")
    json.dump(J, open("reports/uncertainty.json", "w"), indent=2)
UNC_TOTAL_S = time.time() - UNC_T0
TIMINGS["uncertainty"] = UNC_TOTAL_S
info("<b>Cell order to run:</b> Runtime → Run all - Block 11b sits between Block 11 and Block 12 and only needs Blocks 1-11 (trained U-Net, test tiles, labels). "
     "To re-run just Block 11b, re-run Blocks 10 and 11 first, then the four Block 11b code cells in order. No existing cell needs to be changed.")
info(f"<b>Expected extra runtime (measured in this run, {DEVICE.type}):</b> fine-tune {UNC_FT_S:.0f} s + MC inference {UNC_MC_TOT_S:.0f} s ({T_EFF} passes × ~{UNC_PASS_S:.1f} s)"
     + (f" + deep ensemble {UNC_ENS_S:.0f} s" if ENS_RAN else "") + f" → Block 11b total {UNC_TOTAL_S:.0f} s. On CPU, or with a smaller budget, T is reduced automatically and reported.")
_cv = ("Estimated per-pixel predictive uncertainty for a U-Net land-cover classifier of Irish peatlands (Sentinel-2, 7 classes) using Monte Carlo dropout "
       f"({T_EFF} stochastic passes with a dropout-augmented, briefly fine-tuned copy of the trained model), evaluated with rejection curves and reliability/ECE analysis against CORINE labels.")
if ENS_RAN: _cv += " Repeated the estimates with a 3-seed deep ensemble and compared both uncertainty sources."
info("<b>One sentence for the CV, based only on what actually ran:</b> \"" + _cv + "\"")

## Block 12 · CORINE 2012 vs 2018 change detection (real data only)
**What it does:** if a real CORINE 2012 raster is available (found in the session or uploaded in Block 3), warps it onto the same 10 m grid and builds a 7-class and a peat/non-peat **transition matrix** in hectares for 2012 → 2018, plus a change map. If the 2012 file is missing - or the labels are the WorldCover proxy - the block is **skipped** and no change numbers are produced.

**Why it matters:** the ML pipeline maps the *current* state; the transition matrix is the *historical* trend. A trend built from simulated data is worse than none. Use the **same CORINE release** for both years (e.g. both `V2020_20u1`) or the official CLC-Change layer, otherwise revisions show up as fake change. The block warns if release versions differ or if an implausible share of pixels changes class.

> 🎤 **Say to the panel:** *Change detection runs only on real CORINE 2012 and 2018 layers from the same release; the ML model maps the current peat extent and the matrix quantifies 2012-2018 change at CORINE's own resolution and mapping unit.*


In [ ]:
banner("12", "CORINE 2012 vs 2018 change detection", "real CORINE only - the block is skipped (never simulated) when the 2012 file is missing")
import re
for _stale in ("reports/transition_matrix.csv", "reports/transition_matrix_7class.csv", "reports/transition_matrix_peat.csv",
               "reports/06_change_detection.png", "reports/change_detection.json"):
    Path(_stale).unlink(missing_ok=True)       # never ship change results left over from an earlier (or simulated) run

CHANGE = dict(status="skipped", reason="")
if LABEL_MODE != "corine":
    CHANGE["reason"] = "labels are the WorldCover proxy, and a 2012-vs-2018 comparison needs real CORINE rasters for both years"
elif CLC2012_PATH is None:
    CHANGE["reason"] = "no CORINE 2012 raster was found or uploaded in Block 3"

if CHANGE["reason"]:
    warn(f"Change detection <b>skipped</b>: {CHANGE['reason']}. No change figures were produced. To run it, put a file with <code>CLC2012</code> in its name "
         "(same CORINE release as the 2018 file) into <code>/content</code> and re-run Blocks 3 and 12.")
else:
    with stage("change_detection"):
        lab_2012, _ = load_corine(CLC2012_PATH, gb)
        _ver = lambda p: (re.search(r"v\d{4}_\d+u\d+", Path(p).name.lower()) or [None])[0]
        v18, v12 = _ver(label_path), _ver(CLC2012_PATH)
        valid = (lab_2012 != 255) & (lab_raw != 255)
        a_c, b_c = lab_2012[valid].astype(np.int64), lab_raw[valid].astype(np.int64)
        cm12 = np.bincount(a_c * 7 + b_c, minlength=49).reshape(7, 7)
        PX_HA = (CFG["res"] ** 2) / 10_000.0                      # 10 m x 10 m = 0.01 ha
        tm = pd.DataFrame(cm12 * PX_HA, index=[f"{n} (2012)" for n in CLASS_NAMES], columns=[f"{n} (2018)" for n in CLASS_NAMES])
        tm.to_csv("reports/transition_matrix_7class.csv")
        n_valid = int(valid.sum()); changed = float((cm12.sum() - np.trace(cm12)) / max(n_valid, 1))
        pe, pn, kept = int(cm12[PEAT].sum()), int(cm12[:, PEAT].sum()), int(cm12[PEAT, PEAT])
        lost, gained = pe - kept, pn - kept
        stay_non = n_valid - pe - gained
        peat_tm = pd.DataFrame(np.array([[stay_non, gained], [lost, kept]]) * PX_HA,
                               index=["Non-peat (2012)", "Peat bogs (2012)"], columns=["Non-peat (2018)", "Peat bogs (2018)"])
        peat_tm.to_csv("reports/transition_matrix_peat.csv")

    if v18 and v12 and v18 != v12:
        warn(f"CORINE release versions differ (2018: {v18} vs 2012: {v12}) - differences may reflect product revisions, not real land-cover change. "
             "Use the same release for both years, or the official CLC-Change layer.")
    elif not (v18 and v12):
        info("Could not read the release version from the file names - make sure both rasters come from the same CORINE release (e.g. both V2020_20u1).")
    if changed > 0.05:
        warn(f"{changed*100:.1f}% of valid pixels change class between 2012 and 2018 - unusually high for CORINE. Check release versions and alignment before interpreting any of this.")

    html_table([[idx, f"{r.iloc[0]:,.1f}", f"{r.iloc[1]:,.1f}"] for idx, r in peat_tm.iterrows()], ["hectares", "Non-peat (2018)", "Peat bogs (2018)"], width="60%")
    net = gained - lost
    pct = (net / pe * 100) if pe else float("nan")
    html_table([["Peat 2012", f"{pe*PX_HA:,.1f} ha"], ["Peat 2018", f"{pn*PX_HA:,.1f} ha"], ["🔥 Lost (2012→2018)", f"{lost*PX_HA:,.1f} ha"],
                ["🌱 Gained (2012→2018)", f"{gained*PX_HA:,.1f} ha"], ["✅ Persisted", f"{kept*PX_HA:,.1f} ha"],
                ["Net change", f"{net*PX_HA:+,.1f} ha ({pct:+.1f}% of 2012 extent)"]], ["quantity", "value"], width="60%")
    info("CORINE status layers are generalised to a 25 ha minimum mapping unit at 100 m: treat small net changes as indicative, and remember CORINE peat class 412 "
         "mixes intact bog with cut-over and extraction areas that are mapped by other classes.")

    m12 = np.where(valid, (lab_2012 == PEAT).astype(np.float32), np.nan); m18 = np.where(valid, (lab_raw == PEAT).astype(np.float32), np.nan)
    chg = m18 - m12
    g_cmap = plt.get_cmap("Greens").copy(); g_cmap.set_bad("#e5e7eb")
    c_cmap = ListedColormap(["#dc2626", "#f3f4f6", "#16a34a"]); c_cmap.set_bad("#ffffff")
    fig, ax = plt.subplots(1, 3, figsize=(18, 6))
    ax[0].imshow(m12[::PS, ::PS], cmap=g_cmap, vmin=0, vmax=1, interpolation="nearest"); ax[0].set_title(f"Peat bogs 2012 · {pe*PX_HA:,.0f} ha", fontweight="bold")
    ax[1].imshow(m18[::PS, ::PS], cmap=g_cmap, vmin=0, vmax=1, interpolation="nearest"); ax[1].set_title(f"Peat bogs 2018 · {pn*PX_HA:,.0f} ha", fontweight="bold")
    ax[2].imshow(chg[::PS, ::PS], cmap=c_cmap, vmin=-1, vmax=1, interpolation="nearest"); ax[2].set_title("Change 2012→2018 (red = lost, green = gained)", fontweight="bold")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.savefig("reports/06_change_detection.png", dpi=130); plt.show()

    CHANGE.update(status="computed", reason="", files=dict(clc2018=str(label_path), clc2012=str(CLC2012_PATH)), release=dict(clc2018=v18, clc2012=v12),
                  peat_ha=dict(y2012=pe * PX_HA, y2018=pn * PX_HA, lost=lost * PX_HA, gained=gained * PX_HA, persisted=kept * PX_HA, net=net * PX_HA),
                  changed_fraction_all_classes=changed, valid_px=n_valid,
                  note="Real CORINE 2012 and 2018 rasters warped to the 10 m study grid; 100 m / 25 ha MMU generalisation applies.")
    json.dump(CHANGE, open("reports/change_detection.json", "w"), indent=2, default=lambda o: float(o))
    ok(f"Change detection complete in {TIMINGS['change_detection']:.1f}s · saved transition_matrix_7class.csv, transition_matrix_peat.csv, change_detection.json, 06_change_detection.png")


## Block 13 · Run summary & export
**What it does:** prints stage timings, a computed headline (with confidence intervals), honest limitations, zips `reports/` and triggers the download.

**Why it matters:** the reports are generated by the code, so any README numbers can match `metrics.json` exactly; skipped stages (e.g. change detection) are stated as skipped.

> 🎤 **Say to the panel:** *Everything I show is generated by this run, with limitations - and anything that was skipped - stated alongside the results.*


In [ ]:
banner("13", "Run summary & export", "everything below is computed from this run - nothing is typed in by hand")
total = sum(TIMINGS.values())
html_table([[k.replace("_", " ").title(), f"{v:.0f} s", f"{v/total*100:.0f}%"] for k, v in TIMINGS.items()] + [["<b>total (excl. upload wait)</b>", f"<b>{total:.0f} s ({total/60:.1f} min)</b>", ""]],
           ["stage", "time", "share"], width="45%")
if label_source != "corine": warn("These results use PROXY labels (WorldCover). State this clearly if you show them - peat F1 is not a real peat-bog score.")
if MODE == "demo": warn("<b>DEMO MODE</b> - fewer scenes, smaller RF sample and shorter training budgets for a fast live run (the study area is identical in both modes). Do not quote these numbers as final results; set MODE = \"full\" in Block 2 for the reported run.")
if CHANGE["status"] != "computed": warn("<b>Change detection was SKIPPED</b> - " + CHANGE["reason"] + ". No change figures or transition matrices exist in this run's outputs.")
rf_res, un = res["RandomForest"], res["UNet"]
_pt, _lo, _hi = DIFF["macro_f1"]
if un["kappa"] < 0.10: verdict = "the U-Net did NOT converge, so this comparison is invalid"
elif not np.isfinite(_lo): verdict = "confidence interval unavailable (too few test blocks)"
elif _lo <= 0 <= _hi: verdict = "no reliable difference (95% CI of the paired macro-F1 difference spans 0)"
else: verdict = "U-Net better" if _pt > 0 else "Random Forest better"
if budget_hit and un["kappa"] >= 0.10: verdict += "; NOTE the U-Net hit its time budget, so this is a time-boxed comparison, not an architecture verdict"
info(f"<b>Headline (macro F1):</b> RF {fci('RandomForest', 'macro_f1')} vs U-Net {fci('UNet', 'macro_f1')} → {verdict}. "
     f"Peat F1: RF {fci('RandomForest', 'peat_f1')} · U-Net {fci('UNet', 'peat_f1')} (test peat pixels: {rf_res['peat_support_px']:,}; {n_test_blocks} test blocks).")
if res_agg: info(f"<b>At 100 m:</b> macro F1 RF {res_agg['RandomForest']['macro_f1']:.3f} vs U-Net {res_agg['UNet']['macro_f1']:.3f}; peat F1 RF {f(res_agg['RandomForest']['peat_f1'])} vs U-Net {f(res_agg['UNet']['peat_f1'])}.")
LABEL_NOTE = ("CORINE is 100 m with a 25 ha minimum mapping unit (coarse against 10 m imagery), so scores are <b>agreement with CORINE, not field-verified accuracy</b>; check its reference year against the 2018 imagery" if label_source == "corine"
              else f"labels are a PROXY ({scheme}: 10 m WorldCover, no peat class, and its release year differs from the 2018 imagery)")
info("<b>Read these as honest limitations:</b> " + LABEL_NOTE + "; summer 2018 was unusually hot and dry in Ireland, so the composite may not be typical; "
     "one area, one season, one block split with few peat blocks (confidence intervals reflect test-block sampling only, not training randomness or other splits, and neighbouring blocks are correlated); "
     "possible cloud-mask bias over dark wet peat (see Block 6 diagnostic); no field validation; small U-Net trained from scratch. "
     "This is not a peatland-condition assessment.")
_pkg_v = {}
for _mod in ("numpy", "scipy", "pandas", "rasterio", "xarray", "odc", "pystac_client", "sklearn", "torch", "segmentation_models_pytorch"):
    try: _pkg_v[_mod] = str(importlib.import_module(_mod).__version__)
    except Exception: _pkg_v[_mod] = "unknown"
json.dump(dict(timings_s={k: round(v, 1) for k, v in TIMINGS.items()}, python=sys.version.split()[0], torch=torch.__version__, device=str(DEVICE),
               package_versions=_pkg_v, n_scenes=N_SCENES, scene_ids=SCENE_IDS, composite_cached=bool(_cached), s2_processing_baselines=BASELINES, s2_offset_removed_dn=OFFSET,
               cloud_mask_diagnostic=MASK_DIAG, change_detection=CHANGE, deterministic_requested=True,
               unet_epochs_run=len(hist), unet_time_budget_hit=bool(budget_hit), unet_context_px=int(CFG["unet_context_px"]), bootstrap_resamples=int(CFG["boot_n"]), mode=MODE),
          open("reports/run_info.json", "w"), indent=2, default=lambda o: float(o))
shutil.make_archive("peatland_reports", "zip", "reports")
ok("Saved: " + " · ".join(sorted(p.name for p in Path("reports").iterdir())) + " · models/unet_best.pth · peatland_reports.zip")
try:
    from google.colab import files as _f; _f.download("peatland_reports.zip")
except Exception: pass
